# SuReSuite 05 · Available to promise

**You will:** answer a request the way a planner does: *can we give this customer Q units of a
product, or Q units of a material from stock, by this date?* The notebook checks the stock position,
plays the request through your own supply chain (your data, the policies as they are on /policies
now, supplier and production lead times, plant capacity, every existing customer's demand), and
answers yes, or how much by then and when the rest, or a nearby date that works (pull in or push back).
**Time:** about 10 minutes (on a large project, a few minutes per candidate date).
**You need:** nothing for demo mode; with a key, `read:data` and `read:policies` (`read:runs` to use your
saved scenario's horizon). Every user can create that key on /developer.

Nothing here changes your project. The request is added to a **copy** of your data on this machine.

**How a promise is decided.** Your chain is run in `N_WORLDS` possible futures (one random seed each),
with and without the request, on the same draws. A week is promised when, in at least `CONFIDENCE` of
those futures, the request is complete by then; what it costs existing customers is reported beside it,
because an order can be on time and still take material someone else needed later. A make-to-order
request is served after every existing customer row (`SERVE_LAST`), which is what "available to promise"
means. And it is answered twice: with your materials replenished **as your policies are**, and with the
request's materials **planned from the order book** (MRP: ordered one lead time ahead of the need), so
you see what accepting the order and telling purchasing now would change.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Filled in for you when you download this notebook from the app's /developer →
# Notebook tab. Leave it as it is to try the offline demo first.
BASE_URL = "https://wckdrutwkytwcomrlpib.supabase.co/functions/v1/api/v1"
PROJECT_ID = ""          # "" = the first project your key can see
SCENARIO_ID = ""         # a baseline scenario; "" = pick one without disruptions (or create one)
POLICY_VERSION_ID = ""   # a policy snapshot; "" = snapshot the current configuration
SUPPLIER_ID = ""         # a supplier id from your own data, for the disruption examples
MODE = "auto"            # "auto" = live when an API key is found, else the offline demo; "live"; "demo"
ACKNOWLEDGE_WARNINGS = False  # True = run anyway when the pre-run data check only warns (missing values use the platform defaults)

In [ ]:
# @title Setup: packages and API key
# ── SETUP: packages and API key ─────────────────────────────────────────────
# Installs only what is missing (Google Colab already has all of it), then finds
# your API key: the Colab Secrets panel first, then the SURESUITE_API_KEY
# environment variable. MODE = "live" also asks for it with a hidden prompt.
# The key is never printed and never stored in the notebook.
import importlib.util
import os
import subprocess
import sys

_needed = ["requests", "pandas", "matplotlib", "pyarrow", "openpyxl"]
_missing = [p for p in _needed if importlib.util.find_spec(p) is None]
if _missing:
    print("installing:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

API_KEY = None
if MODE != "demo":
    try:
        from google.colab import userdata  # Colab: the Secrets panel (key icon, left sidebar)
        API_KEY = userdata.get("SURESUITE_API_KEY")
    except Exception:
        pass
    API_KEY = API_KEY or os.environ.get("SURESUITE_API_KEY")
    if not API_KEY and MODE == "live":
        from getpass import getpass
        API_KEY = getpass("SuReSuite API key (sk_test_… or sk_live_…): ").strip()
    if API_KEY and not API_KEY.startswith("sk_"):
        raise ValueError("an API key starts with sk_test_ or sk_live_ — create one on the app's /developer page")
    if MODE == "live" and not API_KEY:
        raise ValueError("MODE is \"live\" but no API key was found")

LIVE = bool(API_KEY)
if LIVE:
    print(f"LIVE mode — {API_KEY.split('_')[1]} key, talking to {BASE_URL}")
else:
    print("DEMO mode — no API key found, so this notebook replays recorded engine output for the\n"
          "Example project (1 product, 2 materials, 3 suppliers). Every cell runs the same way in\n"
          "live mode. To go live, add SURESUITE_API_KEY (see the Setup notes) and run all cells again.")

In [ ]:
# @title SuReSuite notebook library (run it; no need to read it)
# ── SuReSuite notebook library ──────────────────────────────────────────────
# The `suresuite` Python package's client (python/suresuite/client.py), inlined
# into each notebook by scripts/notebooks/build-notebooks.mjs. Run this cell;
# you do not need to read it. It defines:
#   connect()                      the API client (live, or the offline demo), checked
#                                  against the scopes the notebook needs
#   api.run_and_wait(...)          dispatch a simulation run and poll it to the end
#   kpi_table / replications_frame / paired_compare / assert_disruption_applied
#   editing_policies(...)          edit policies and ALWAYS put them back
#   plot_kpis / plot_series        small, honest charts
# Every helper talks to the public /v1 API the same way in live and demo mode.
import base64
import contextlib
import copy
import hashlib
import io
import json
import math
import re
import time
import uuid
import zlib

import pandas as pd
import requests

try:  # the charts need matplotlib; the package works without it (`pip install suresuite[plots]`)
    import matplotlib.pyplot as plt
except ImportError:  # pragma: no cover - exercised only without the extra
    plt = None

try:  # as the `suresuite` package; in a notebook the KPI_DISPLAY cell is inlined below instead
    from .kpi_display import KPI_DISPLAY
except ImportError:
    pass

FAMILIES = ("sourcing", "inventory", "transport", "fulfillment", "production", "recovery", "demand")
TERMINAL = ("done", "failed", "cancelled")  # a run's lifecycle: queued → running → done | failed | cancelled
PALETTE = ["#2a78d6", "#eda100", "#008300", "#e87ba4", "#7a5af8", "#5f6b7a"]  # fixed order per series
T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262,
       10: 2.228, 11: 2.201, 12: 2.179, 13: 2.16, 14: 2.145, 15: 2.131, 16: 2.12, 17: 2.11,
       18: 2.101, 19: 2.093, 20: 2.086, 21: 2.08, 22: 2.074, 23: 2.069, 24: 2.064, 25: 2.06,
       26: 2.056, 27: 2.052, 28: 2.048, 29: 2.045, 30: 2.042}
_MONEY = {"symbol": "€"}  # replaced by the engine's declared unit when the catalog is read
KEY_FIELDS = ("key_prefix", "env", "principal", "scopes", "project_ids")  # GET /key, as the gateway answers it


class SuReSuiteError(RuntimeError):
    """The API's error envelope {"error": {"code", "message", "details"}} as an exception."""

    def __init__(self, status, code, message, details=None, request_id=None):
        hint = ""
        if code == "missing_scope":  # a key's scopes are fixed when it is created
            hint = (" — create a key that has this scope on the app's /developer page"
                    " (Create API key), then set SURESUITE_API_KEY to it and run all cells again")
        super().__init__(f"[{status} {code}] {message}{hint}")
        self.status, self.code, self.message = status, code, message
        self.details, self.request_id = details, request_id


class Stop(Exception):
    """A notebook cannot go on, and says why and what to do. In Jupyter and Colab it
    shows as that one message rather than a traceback (`%tb` still has the trace)."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


# ── transports: how a request reaches the platform ──────────────────────────
class HttpTransport:
    """The real thing: HTTPS to the /v1 gateway with your key."""

    def __init__(self, base_url, api_key, timeout=60):
        self.base = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers["Authorization"] = f"Bearer {api_key}"

    def request(self, method, path, params=None, body=None, headers=None):
        r = self.session.request(method, self.base + path, params=params, json=body,
                                 headers=headers or {}, timeout=self.timeout)
        try:
            payload = r.json() if r.content else {}
        except ValueError:  # an HTML error page from a proxy, say
            payload = {"error": {"code": "non_json_response", "message": r.text[:200]}}
        return r.status_code, dict(r.headers), payload

    def fetch(self, url):
        # A signed storage URL carries its own authorization: never send the API key to it.
        r = requests.get(url, timeout=self.timeout)
        r.raise_for_status()
        return r.content


class SuReSuite:
    """Client for the SuReSuite public /v1 API."""

    def __init__(self, transport, live):
        self.t, self.live = transport, live
        self._catalog = {}

    # ── plumbing ──
    def _call(self, method, path, params=None, body=None, headers=None, retries=4):
        for attempt in range(retries + 1):
            try:
                status, hdrs, payload = self.t.request(method, path, params=params, body=body, headers=headers)
            except requests.RequestException as e:  # network trouble: retry with backoff, then give up
                if attempt == retries:
                    raise
                wait = 2 ** (attempt + 1)
                print(f"  network error ({type(e).__name__}) — retrying in {wait}s")
                time.sleep(wait)
                continue
            err = (payload or {}).get("error") or {} if status >= 400 else {}
            if status == 429 and err.get("code") in ("rate_limited", "too_many_failed_auths") and attempt < retries:
                wait = min(int(hdrs.get("Retry-After", "5") or 5), 60)
                print(f"  rate-limited — waiting {wait}s")
                time.sleep(wait)
                continue
            if status in (502, 503, 504) and attempt < retries:
                time.sleep(2 ** (attempt + 1))
                continue
            remaining = hdrs.get("X-RateLimit-Remaining")
            if remaining is not None and remaining.isdigit() and int(remaining) <= 5:
                print(f"  note: {remaining} requests left in this key's rate-limit window")
            if status >= 400:
                raise SuReSuiteError(status, err.get("code", "unknown"), err.get("message", ""),
                                     err.get("details"), hdrs.get("X-Request-Id"))
            return payload
        raise RuntimeError("unreachable")

    def get(self, path, **params):
        return self._call("GET", path, params={k: v for k, v in params.items() if v is not None} or None)

    def post(self, path, body=None, idempotency_key=None):
        headers = {"Idempotency-Key": idempotency_key} if idempotency_key else None
        return self._call("POST", path, body=body or {}, headers=headers)

    def put(self, path, body):
        return self._call("PUT", path, body=body)

    def list_all(self, path, limit=100, **params):
        """Follow cursor pagination (?limit=&cursor=) until the last page."""
        rows, cursor = [], None
        while True:
            page = self.get(path, limit=limit, cursor=cursor, **params)
            rows += page["data"]
            cursor = page.get("next_cursor")
            if not cursor:
                return rows

    # ── the key itself ──
    def key(self):
        """The calling key as the platform sees it now: `scopes`, `env`, `principal`
        and `project_ids`. None from a gateway that predates `GET /key`."""
        try:
            return self.get("/key")
        except SuReSuiteError as e:
            if e.code == "route_not_found":
                return None
            raise

    # ── projects and input data ──
    def projects(self):
        return self.list_all("/projects")

    def project(self, project_id):
        return self.get(f"/projects/{project_id}")

    def dataset_versions(self, project_id):
        return self.get(f"/projects/{project_id}/dataset-versions")["data"]

    def freeze_dataset(self, project_id, label=None):
        return self.post(f"/projects/{project_id}/datasets:freeze", {"label": label} if label else {})

    # ── policies ──
    def catalog(self, project_id):
        if project_id not in self._catalog:
            cat = self.get(f"/projects/{project_id}/policy-catalog")
            money = {k["unit"] for k in cat.get("kpis") or [] if k.get("name") in ("revenue", "lost_sales_value")}
            if len(money) == 1:
                _MONEY["symbol"] = money.pop()
            self._catalog[project_id] = cat
        return self._catalog[project_id]

    def policies(self, project_id):
        return self.get(f"/projects/{project_id}/policies")

    def policy_defaults(self, project_id):
        """The seven policy families, each a dict ({} = engine defaults)."""
        d = self.policies(project_id)["defaults"] or {}
        return {f: dict(d.get(f) or {}) for f in FAMILIES}

    def set_policy(self, project_id, family, **fields):
        """Change some fields of ONE family. PUT replaces a family as a whole, so this
        reads the family, merges your fields in, and writes the whole family back."""
        if family not in FAMILIES:
            raise ValueError(f"family must be one of {FAMILIES}")
        merged = {**self.policy_defaults(project_id)[family], **fields}
        self.put(f"/projects/{project_id}/policies", {"defaults": {family: merged}})
        return merged

    def snapshot_policies(self, project_id, label):
        """An immutable policy version. An unchanged configuration returns the version
        that already holds it (content-addressed), so the label may be an older one."""
        return self.post(f"/projects/{project_id}/policy-versions", {"label": label})

    def dataset_version(self, project_id, version="latest", tables=None):
        """One frozen dataset version WITH its rows (`snapshot`): exactly what a run
        reads. `version` is an id or "latest"; `tables` narrows it."""
        return self.get(f"/projects/{project_id}/dataset-versions/{version}",
                        tables=",".join(tables) if tables else None)

    def policy_version(self, project_id, version="latest"):
        """One frozen policy version with its `snapshot` and `policy_hash`."""
        return self.get(f"/projects/{project_id}/policy-versions/{version}")

    def engine(self, version=None):
        """The engine's wheels: sha256, size and a short-lived signed URL each.

        ``version`` — ``"0.4.0"`` (the newest build of that version) or an exact
        build ``"scsim-0.4.0+<digest>"`` — fetches an EARLIER engine, so a stored
        result can be re-run on the engine that produced it (PLAN.md §25 WP 15.3)."""
        return self.get("/engine", version=version)

    def policy_versions(self, project_id):
        return self.get(f"/projects/{project_id}/policy-versions")["data"]

    # ── scenarios ──
    def scenarios(self, project_id):
        return self.list_all(f"/projects/{project_id}/scenarios")

    def create_scenario(self, project_id, name, **fields):
        return self.post(f"/projects/{project_id}/scenarios", {"name": name, **fields})

    # ── runs ──
    def run(self, run_id):
        return self.get(f"/runs/{run_id}")

    def replications(self, run_id):
        return self.list_all(f"/runs/{run_id}/replications")

    def validation(self, run_id):
        return self.get(f"/runs/{run_id}/validation")

    def cancel(self, run_id):
        return self.post(f"/runs/{run_id}:cancel")

    def series(self, run_id, expired_ok=False):
        """Every weekly series of a run as ONE long table:
        rep_index · model_rep · event_rep · week · one column per series.
        `expired_ok=True` returns None (and says so) for a run whose series have expired."""
        page = self.get(f"/runs/{run_id}/replications", include="time_series", limit=100)
        where = page.get("series") or {}
        if where.get("location") == "object" and where.get("url"):
            blob = self.t.fetch(where["url"])  # live: Parquet bytes; demo: a column table
            return pd.read_parquet(io.BytesIO(blob)) if isinstance(blob, (bytes, bytearray)) else pd.DataFrame(blob)
        if where.get("location") == "expired":
            if expired_ok:
                print(f"run {run_id}: its weekly series have expired (the KPIs above are kept) — dispatch it again "
                      "with force_rerun=True to compute them")
                return None
            raise SuReSuiteError(410, "series_expired",
                                 "this run's weekly series have expired; re-run its RunKey to reproduce them",
                                 where)
        rows = []  # an older run keeps its series on each replication row
        for rep in self.list_all(f"/runs/{run_id}/replications", include="time_series"):
            ts = rep.get("time_series") or {}
            for week in range(max((len(v) for v in ts.values()), default=0)):
                rows.append({"rep_index": rep["rep_index"], "model_rep": (rep.get("kpis") or {}).get("model_rep"),
                             "event_rep": (rep.get("kpis") or {}).get("event_rep"), "week": week,
                             **{k: (v[week] if week < len(v) else None) for k, v in ts.items()}})
        return pd.DataFrame(rows)

    def dispatch(self, project_id, scenario_id, policy_version_id, *, acknowledge_warnings=False,
                 force_rerun=False, idempotency_key=None):
        """POST a run. Returns the run row (a reused one on 409, if you allow reuse)."""
        body = {"scenario_id": scenario_id, "policy_version_id": policy_version_id}
        if acknowledge_warnings:
            body["acknowledge_warnings"] = True
        if force_rerun:
            body["force_rerun"] = True
        # The key is derived from WHAT you ask for: re-running this cell replays the
        # same run, and changing any input dispatches a new one.
        key = idempotency_key or "nb-" + hashlib.sha256(
            json.dumps({"project": project_id, **body}, sort_keys=True).encode()).hexdigest()[:40]
        try:
            submitted = self.post(f"/projects/{project_id}/runs", body, idempotency_key=key)
        except SuReSuiteError as e:
            if e.code == "reuse_available":
                cand = (e.details or {}).get("reuse_candidate") or {}
                print(f"identical completed run already exists — reusing {cand.get('run_id')} "
                      f"(pass force_rerun=True to compute it again)")
                return self.run(cand["run_id"])
            if e.code == "validation_failed":
                d = e.details or {}
                print("the pre-run data gate stopped this run:")
                for f in d.get("findings") or []:
                    print(f"  - [{f.get('severity')}] {f.get('message')}")
                if d.get("validation") == "ack_required":
                    print("these are warnings: if they are acceptable, set ACKNOWLEDGE_WARNINGS = True in the CONFIG cell "
                          "(or pass acknowledge_warnings=True) and run this cell again")
            raise
        print(f"run {submitted['run_id']} dispatched ({submitted['status']})")
        return self.run(submitted["run_id"])

    def wait(self, run_id, timeout_seconds=1800):
        """Poll a run until it is done, failed or cancelled (backoff 3 s → 30 s)."""
        started, delay = time.time(), 3.0
        while True:
            run = self.run(run_id)
            if run["status"] in TERMINAL:
                print(f"\n→ {run['status']} after {time.time() - started:.0f}s"
                      f" — {run.get('rep_count_done') or 0} replications")
                if run["status"] == "failed":
                    print("error:", run.get("error_message"))
                return run
            done, target = run.get("rep_count_done") or 0, run.get("rep_count_target") or "?"
            print(f"  {run['status']} — replication {done}/{target}   ", end="\r")
            if time.time() - started > timeout_seconds:
                raise TimeoutError(f"run {run_id} still {run['status']} after {timeout_seconds}s")
            if self.live:
                time.sleep(delay)
                delay = min(delay * 1.5, 30.0)

    def run_and_wait(self, project_id, scenario_id, policy_version_id, **kw):
        run = self.dispatch(project_id, scenario_id, policy_version_id, **kw)
        return run if run["status"] in TERMINAL else self.wait(run["id"])


# ── results as tables ───────────────────────────────────────────────────────
def _fmt(kind, v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    if kind == "pct2":
        return f"{v * 100:.2f}%"
    if kind == "pct1":
        return f"{v * 100:.1f}%"
    if kind == "money":
        return f"{'-' if v < 0 else ''}{_MONEY['symbol']}{abs(v):,.0f}"
    if kind == "units":
        return f"{v:,.0f}"
    if kind.startswith("fixed"):
        return f"{v:.{int(kind[5:] or 2)}f}"
    return f"{v:,.4g}"


def kpi_label(key):
    return KPI_DISPLAY.get(key, (key, "num", None))[0]


def fmt_kpi(key, value):
    return _fmt(KPI_DISPLAY.get(key, (key, "num", None))[1], value)


def _numeric(d):
    return {k: v for k, v in (d or {}).items()
            if not k.startswith("_") and isinstance(v, (int, float)) and not isinstance(v, bool)
            and not math.isnan(v)}


def _order(keys):
    rank = {k: i for i, k in enumerate(KPI_DISPLAY)}
    return sorted(keys, key=lambda k: (rank.get(k, 10_000), k))


def kpi_table(run, kpis=None):
    """A run's aggregate KPIs: mean ± 95 % CI half-width, labelled and formatted the way
    the Simulation Lab shows them. Measures the engine did not produce are left out."""
    agg, ci = _numeric(run.get("aggregate_kpis")), run.get("ci_half_widths") or {}
    keys = [k for k in (kpis or _order(agg)) if k in agg]
    return pd.DataFrame([{"kpi": k, "measure": kpi_label(k), "mean": fmt_kpi(k, agg[k]),
                          "± 95% CI": fmt_kpi(k, ci.get(k)) if ci.get(k) is not None else "—",
                          "value": agg[k]} for k in keys],
                        columns=["kpi", "measure", "mean", "± 95% CI", "value"]).set_index("kpi")


def replications_frame(replications):
    """One row per replication: rep_index, model_rep, event_rep and every KPI."""
    rows = [{"rep_index": r["rep_index"], **(r.get("kpis") or {})} for r in replications]
    return pd.DataFrame(rows, columns=None if rows else ["rep_index"]).set_index("rep_index").sort_index()


def capacity_binding(run):
    """Which products / suppliers capacity held back, if any (from the run's _meta)."""
    return ((run.get("aggregate_kpis") or {}).get("_meta") or {}).get("capacity_binding")


def assert_disruption_applied(replications):
    """A disruption that reached the engine leaves recovery measures on every replication.
    None of them means every event was SKIPPED (a target the engine cannot disrupt) and
    the run is really a baseline — say so instead of reporting its KPIs as a stress test."""
    if not any("recovery_measurable" in (r.get("kpis") or {}) for r in replications):
        raise AssertionError(
            "no disruption reached the engine: every event was skipped. A target must be one of "
            "this project's supplier ids or the plant (\"plant\") — material, customer and lane "
            "targets are not supported by the engine yet.")
    print("disruption applied — recovery measures are present on the replications")


def paired_compare(reps_a, reps_b, kpis=None, labels=("A", "B")):
    """B − A per KPI, paired replication by replication on the common-random-numbers
    cell (model_rep, event_rep) — the Simulation Lab's Compare rule. A direction
    ("better" / "worse") is drawn only when the 95 % interval of the paired
    difference excludes zero; otherwise the honest answer is "no clear difference"."""

    def key(r):
        k = r.get("kpis") or {}
        return (f"m{int(k['model_rep'])}:e{int(k.get('event_rep') or 0)}"
                if isinstance(k.get("model_rep"), (int, float)) else f"i{r['rep_index']}")

    a_by = {key(r): r for r in reps_a if r.get("status", "done") == "done"}
    keys = kpis or _order(set().union(*[_numeric(r.get("kpis")).keys() for r in reps_a])
                          - {"model_rep", "event_rep", "recovery_measurable", "ttr_censored", "tts_censored"})
    rows = []
    for kpi in keys:
        diffs, va_all, vb_all = [], [], []
        for r in reps_b:
            a = a_by.get(key(r))
            va = _numeric((a or {}).get("kpis")).get(kpi)
            vb = _numeric(r.get("kpis")).get(kpi)
            if a is not None and va is not None and vb is not None and r.get("status", "done") == "done":
                diffs.append(vb - va)
                va_all.append(va)
                vb_all.append(vb)
        n = len(diffs)
        if n < 2:
            continue
        m = sum(diffs) / n
        sd = math.sqrt(sum((d - m) ** 2 for d in diffs) / (n - 1))
        hw = (T95.get(n - 1, 1.96) * sd) / math.sqrt(n)
        separated = abs(m) > hw
        direction = KPI_DISPLAY.get(kpi, (kpi, "num", None))[2]
        if not separated:
            verdict = "no clear difference"
        elif direction is None:
            verdict = "higher" if m > 0 else "lower"
        else:
            verdict = f"{labels[1]} better" if (m > 0) == direction else f"{labels[1]} worse"
        rows.append({"kpi": kpi, "measure": kpi_label(kpi),
                     labels[0]: fmt_kpi(kpi, sum(va_all) / n), labels[1]: fmt_kpi(kpi, sum(vb_all) / n),
                     f"{labels[1]} − {labels[0]}": f"{'+' if m >= 0 else '-'}{fmt_kpi(kpi, abs(m))} ± {fmt_kpi(kpi, hw)}",
                     "pairs": n, "verdict": verdict, "delta": m, "half_width": hw})
    return pd.DataFrame(rows).set_index("kpi") if rows else pd.DataFrame()


@contextlib.contextmanager
def editing_policies(api, project_id):
    """Edit the project's live policy defaults inside this block; they are put back
    exactly as they were when the block ends — even if a run fails. (Per-node
    overrides are not touched: the API has no way to delete one.)"""
    original = api.policy_defaults(project_id)
    try:
        yield original
    finally:
        api.put(f"/projects/{project_id}/policies", {"defaults": original})
        print("policies restored to their original values")


# ── picking what to work on ─────────────────────────────────────────────────
FRAME_FIELDS = ("horizon_days", "warmup_days", "replications", "seed", "crn", "primary_kpi")


def pick_project(api, project_id=""):
    """The configured project, or the first one this key can see."""
    projects = api.projects()
    if not projects:
        raise Stop("this key can see no projects — check its organization and project restriction on /developer")
    if project_id and project_id not in {p["id"] for p in projects}:
        raise Stop(f"PROJECT_ID {project_id} is not a project this key can see — set PROJECT_ID = \"\" in the "
                   "CONFIG cell to use the first one, or download the notebook again from /developer")
    project = api.project(project_id) if project_id else projects[0]
    print(f"project: {project['name']}  ({project['id']})")
    return project


def baseline_scenario(api, project_id, scenario_id=""):
    """The configured scenario, else the newest one WITHOUT disruptions, else a new
    52-week baseline (the Simulation Lab's defaults: 364 days, 10 replications,
    seed 42, common random numbers on)."""
    scenarios = api.scenarios(project_id)
    if scenario_id:
        sc = next((s for s in scenarios if s["id"] == scenario_id), None)
        if sc is None:
            raise Stop(f"SCENARIO_ID {scenario_id} is not a scenario of project {project_id} — set "
                       "SCENARIO_ID = \"\" in the CONFIG cell to pick or create a baseline")
    else:
        sc = next((s for s in scenarios if not s.get("disruption_schedule")), None)
        if sc is None:
            created = api.create_scenario(project_id, "Notebook baseline", description="52 weeks, no disruption",
                                          horizon_days=364, replications=10, seed=42, crn=True)
            sc = next(s for s in api.scenarios(project_id) if s["id"] == created["id"])
            print("created a baseline scenario")
    if sc.get("disruption_schedule"):
        print("note: this scenario already has disruptions — it is not a clean baseline")
    if (sc.get("horizon_days") or 0) < 364:
        print(f"note: horizon_days={sc.get('horizon_days')} runs as 52 weeks — the engine's minimum horizon")
    print(f"scenario: {sc['name']}  ({sc['id']}) — {sc['horizon_days']} days, {sc['replications']} replications, "
          f"seed {sc['seed']}, CRN {'on' if sc.get('crn') else 'off'}")
    return sc


def baseline_policy_version(api, project_id, version_id="", label="notebook baseline"):
    """The configured policy version, else a snapshot of the current policies (an
    unchanged configuration returns the version that already holds it), else — for a
    key without `write:policies` — the newest saved version, and it says so."""
    versions = api.policy_versions(project_id)
    if version_id:
        v = next((v for v in versions if v["id"] == version_id), None)
        if v is None:
            raise Stop(f"POLICY_VERSION_ID {version_id} is not a policy version of project {project_id} — set "
                       "POLICY_VERSION_ID = \"\" in the CONFIG cell to use the current policies")
        return v
    try:
        return api.snapshot_policies(project_id, label)
    except SuReSuiteError as e:
        if e.code != "missing_scope":
            raise
    if not versions:
        raise Stop("this project has no saved policy version and this key cannot save one (no write:policies) "
                   "— save one in the app (/policies → Save version) and run again")
    newest = max(versions, key=lambda v: v.get("created_at") or "")
    print(f"note: this key cannot snapshot policies (no write:policies), so the newest saved version is used: "
          f"{newest.get('label') or 'unlabelled'} — it may differ from the policies as they are now")
    return newest


def require_done(run):
    """Stop here, with the reason, unless the run finished."""
    if run["status"] != "done":
        raise Stop(f"run {run['id']} ended {run['status']}"
                   + (f": {run.get('error_message')}" if run.get("error_message") else "")
                   + " — nothing below has results to read")
    return run


def stress_scenario(api, project_id, base, events, name):
    """`base`'s frame (horizon, replications, seed, CRN) plus a disruption schedule.
    Reuses an existing scenario with the same name and schedule, so running a cell
    again does not pile up copies in the Simulation Lab."""
    norm = lambda sched: _canon([{k: e[k] for k in ("target", "start_day", "duration_days", "magnitude_pct")}
                                 for e in sched or []])
    for sc in api.scenarios(project_id):
        if sc["name"] == name and norm(sc.get("disruption_schedule")) == norm(events):
            return sc
    created = api.create_scenario(project_id, name, description="created by a SuReSuite notebook",
                                  disruption_schedule=events,
                                  **{k: base[k] for k in FRAME_FIELDS if base.get(k) is not None})
    return next(s for s in api.scenarios(project_id) if s["id"] == created["id"])


def outage(target, start_day=140, duration_days=42, magnitude_pct=100):
    """One disruption event. target: a supplier id of the project, or "plant".
    magnitude_pct 100 = the target ships nothing for the window (its deliveries are
    deferred, not lost); below 100 = its capacity is cut BY that share, which bites
    only when the remaining capacity is below what the plan needs."""
    return {"target": target, "target_type": "node", "start_day": start_day,
            "duration_days": duration_days, "magnitude_pct": magnitude_pct}


# ── the run-results workbook (the app's "verifiable export") ────────────────
def export_run_workbook(api, run, scenario, path, policy_version_label=None):
    """Write the sheets the app's run-results workbook carries — run_meta,
    aggregate_kpis, replication_kpis, series_<key> (weeks × replications) and
    reproducibility — so a figure that leaves the notebook says what produced it."""
    reps = [r for r in api.replications(run["id"]) if r.get("status", "done") == "done"]
    agg, ci = run.get("aggregate_kpis") or {}, run.get("ci_half_widths") or {}
    rng = agg.get("_range") or {}
    validation = api.validation(run["id"])
    meta = [
        ("run_id", run["id"]), ("status", run["status"]), ("created_at", run.get("created_at")),
        ("ended_at", run.get("ended_at")), ("code_version (engine)", run.get("code_version")),
        ("policy_version_id", run.get("policy_version_id")), ("policy_version_label", policy_version_label),
        ("policy_hash (SHA-256)", run.get("policy_hash")), ("dataset_version_id", run.get("dataset_version_id")),
        ("graph_hash (SHA-256)", run.get("graph_hash")), ("scenario_hash (baseline fingerprint)", run.get("scenario_hash")),
        ("model_validation_id", run.get("model_validation_id")), ("scenario_id", scenario["id"]),
        ("scenario_name", scenario["name"]), ("root seed", scenario.get("seed")),
        ("replications (target)", run.get("rep_count_target")), ("replications (done)", run.get("rep_count_done")),
        ("CRN (common random numbers)", scenario.get("crn")), ("horizon_days", scenario.get("horizon_days")),
        ("warmup_detected_at (weeks, engine)", run.get("warmup_detected_at")),
        ("disruption_schedule (full JSON; empty = baseline)", json.dumps(scenario.get("disruption_schedule") or [])),
        ("credibility", validation.get("status")), ("source", "SuReSuite public API /v1" if api.live else "offline demo"),
        ("generated", pd.Timestamp.now(tz="UTC").isoformat()),
    ]
    aggregate = pd.DataFrame([{"kpi": k, "mean": agg[k], "ci_halfwidth": ci.get(k),
                               "min": (rng.get(k) or {}).get("min"), "max": (rng.get(k) or {}).get("max")}
                              for k in sorted(_numeric(agg))])
    replication = pd.DataFrame([{"rep_index": r["rep_index"], "seed_used (root seed)": r.get("seed_used"),
                                 **(r.get("kpis") or {})} for r in reps])
    series = api.series(run["id"])
    label = lambda row: f"rep {int(row.rep_index)}" + (f" · world {int(row.model_rep)}" if pd.notna(row.model_rep) else "")
    series["replication"] = series.apply(label, axis=1)
    repro = [
        ("dataset", f"dataset version {run.get('dataset_version_id')} · graph_hash {run.get('graph_hash')}"),
        ("policy", f"policy version {run.get('policy_version_id')} · policy_hash {run.get('policy_hash')}"),
        ("scenario", f"scenario {scenario['id']} · scenario_hash {run.get('scenario_hash')} · seed {scenario.get('seed')}"
                     f" · disruptions {json.dumps(scenario.get('disruption_schedule') or [])}"),
        ("engine", run.get("code_version")),
        ("known limit", "assembled by a notebook from the API's run row; network-analysis figures and "
                        "mapping warnings are not part of this record (the API does not return them)"),
    ]
    with pd.ExcelWriter(path) as xl:
        pd.DataFrame(meta, columns=["field", "value"]).to_excel(xl, sheet_name="run_meta", index=False)
        aggregate.to_excel(xl, sheet_name="aggregate_kpis", index=False)
        replication.to_excel(xl, sheet_name="replication_kpis", index=False)
        for col in [c for c in series.columns if c not in ("rep_index", "model_rep", "event_rep", "week", "replication")]:
            series.pivot_table(index="week", columns="replication", values=col, sort=False).to_excel(
                xl, sheet_name=f"series_{col}"[:31])
        pd.DataFrame(repro, columns=["binding", "value"]).to_excel(xl, sheet_name="reproducibility", index=False)
    print(f"wrote {path}: run_meta, aggregate_kpis, replication_kpis, {series.shape[1] - 5} series sheets, reproducibility")
    return path


# ── charts ──────────────────────────────────────────────────────────────────
def _need_plt():
    if plt is None:
        raise ImportError("charts need matplotlib: pip install 'suresuite[plots]'")


def plot_kpis(runs, kpis, title=None):
    """One small panel per KPI (their scales differ), one bar per run, with the
    run's 95 % CI half-width as the error bar. `runs` is {label: run_row}."""
    _need_plt()
    kpis = [k for k in kpis if all(k in _numeric(r.get("aggregate_kpis")) for r in runs.values())]
    if not kpis:
        print("none of those KPIs is present on every run")
        return
    fig, axes = plt.subplots(1, len(kpis), figsize=(2.9 * len(kpis), 3.3), squeeze=False)
    for ax, kpi in zip(axes[0], kpis):
        labels = list(runs)
        vals = [_numeric(runs[l].get("aggregate_kpis"))[kpi] for l in labels]
        errs = [(runs[l].get("ci_half_widths") or {}).get(kpi) or 0 for l in labels]
        ax.bar(range(len(labels)), vals, yerr=errs, capsize=3, width=0.65,
               color=[PALETTE[i % len(PALETTE)] for i in range(len(labels))])
        ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right", fontsize=8)
        ax.set_title(kpi_label(kpi), fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
    if title:
        fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_series(series_by_label, column, title=None, window=None, warmup_week=None):
    """Mean weekly trajectory per run with a band from the lowest to the highest
    replication. `series_by_label` is {label: api.series(run_id)}; `window` is a
    (start_week, end_week) disruption window to shade."""
    _need_plt()
    if not any(df is not None for df in series_by_label.values()):
        print("no weekly series to plot")
        return
    fig, ax = plt.subplots(figsize=(8, 3.4))
    for i, (label, df) in enumerate(series_by_label.items()):
        if df is None or column not in df:
            continue
        g = df.groupby("week")[column]
        color = PALETTE[i % len(PALETTE)]
        ax.plot(g.mean().index, g.mean().values, color=color, linewidth=2, label=label)
        ax.fill_between(g.min().index, g.min().values, g.max().values, color=color, alpha=0.12, linewidth=0)
    if window:
        ax.axvspan(window[0], window[1], color="#5f6b7a", alpha=0.10, label="disruption window")
    if warmup_week:
        ax.axvline(warmup_week, color="#5f6b7a", linestyle=":", linewidth=1, label="warm-up ends")
    ax.set_xlabel("week")
    ax.set_ylabel(column)
    ax.set_title(title or column, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False, loc="best")
    plt.tight_layout()
    plt.show()


def effective_window(event, run):
    """The weeks an event actually covered: the engine works in weeks, starts no
    earlier than week 1, and moves an event that would start inside the warm-up to
    the warm-up's end."""
    start = max(1, round(event["start_day"] / 7))
    start = max(start, int(run.get("warmup_detected_at") or 0))
    return start, start + max(1, min(52, round(event["duration_days"] / 7)))


# ── the offline demo: the same API, answered from recorded engine output ─────
RUN_FIELDS = ("id", "scenario_id", "project_id", "status", "started_at", "ended_at", "created_at",
              "updated_at", "rep_count_target", "rep_count_done", "aggregate_kpis", "ci_half_widths",
              "warmup_detected_at", "policy_version_id", "policy_hash", "dataset_version_id",
              "graph_hash", "scenario_hash", "model_validation_id", "gate_skipped", "error_message",
              "code_version")
SCENARIO_FIELDS = {"name": None, "description": "", "horizon_days": 90, "warmup_days": 14,
                   "replications": 10, "seed": 42, "crn": True, "disruption_schedule": [],
                   "recovery_overrides": {}, "primary_kpi": "fill_rate"}
RUN_BODY_FIELDS = ("scenario_id", "policy_version_id", "acknowledge_warnings", "force_rerun")


def _canon(v):
    return json.dumps(v, sort_keys=True, separators=(",", ":"))


def _sha(v):
    return hashlib.sha256(_canon(v).encode()).hexdigest()


def _demo_key(policies, schedule, frame):
    """MUST match demo_key() in notebooks/tools/record_demo.py (pinned by the tests)."""
    pol = {f: v for f, v in sorted(policies.items()) if v}
    sched = sorted(({"target": str(e["target"]), "start_day": int(e["start_day"]),
                     "duration_days": int(e["duration_days"]),
                     "magnitude_pct": float(e.get("magnitude_pct", 100))} for e in schedule), key=_canon)
    fr = {k: frame[k] for k in ("horizon_days", "replications", "seed", "crn")}
    return _sha({"policies": pol, "schedule": sched, "frame": fr})


class DemoTransport:
    """Answers the /v1 routes the notebooks use from recorded engine output, keeping
    policies, scenarios and runs in memory for this session. Responses have the live
    API's shapes; a request it has no recording for is refused, never invented."""

    TS = "2026-10-01T00:00:00+00:00"

    def __init__(self, data):
        self.d = data
        self.pid = data["project"]["id"]
        self.recordings = {r["key"]: r for r in data["recordings"]}
        self.supplier_ids = {s["supplier_id"] for s in data["dataset"]["suppliers"]}
        self.defaults = {f: {} for f in FAMILIES}
        self.overrides = []
        self.datasets = [{"id": self._id("dataset", 1), "label": "initial upload", "graph_hash": data["graph_hash"],
                          "author_email": "demo@example.com", "created_at": self.TS, "version_no": 1, "tuple": None}]
        self.versions, self.scenarios, self.runs, self.idem = [], [], {}, {}
        self._new_scenario({"name": "Baseline (demo)", "description": "52 weeks, no disruption",
                            "horizon_days": 364, "replications": 10})
        self._n = 0

    def _id(self, kind, n):
        return str(uuid.uuid5(uuid.NAMESPACE_URL, f"suresuite-demo/{kind}/{n}"))

    def _err(self, status, code, message, details=None):
        return status, {"X-Request-Id": "demo"}, {"error": {"code": code, "message": message,
                                                            **({"details": details} if details else {})}}

    def _ok(self, body, status=200):
        return status, {"X-Request-Id": "demo"}, body

    def _new_scenario(self, body):
        unknown = set(body) - set(SCENARIO_FIELDS)
        if unknown:
            raise ValueError(f"unknown scenario field(s): {sorted(unknown)}")
        sc = {**copy.deepcopy(SCENARIO_FIELDS), **copy.deepcopy(body),
              "id": self._id("scenario", len(self.scenarios) + 1), "created_at": self.TS, "updated_at": self.TS}
        self.scenarios.insert(0, sc)
        return sc

    def fetch(self, url):
        rec = self.recordings[self.runs[url.split("demo://series/")[1]]["_key"]]
        return rec["series"]  # already a column table; read by series() below

    def request(self, method, path, params=None, body=None, headers=None):
        params, body, headers = params or {}, body or {}, headers or {}
        p = path.split("?")[0]
        m = re.fullmatch(r"/projects/([^/]+)(/.*)?", p)
        if p == "/projects" and method == "GET":
            return self._ok({"data": [self.d["project"]], "next_cursor": None})
        if m:
            if m.group(1) != self.pid:
                return self._err(404, "project_not_found", "project not found")
            return self._project(method, m.group(2) or "", body, headers)
        if p == "/key" and method == "GET":
            return self._ok({"key_prefix": "demo", "env": "test", "principal": "personal",
                             "scopes": [f"{a}:{b}" for b in ("data", "policies", "runs") for a in ("read", "write")],
                             "project_ids": None})
        if p == "/engine" and method == "GET":
            return self._err(403, "demo_no_engine",
                             "the offline demo cannot hand out the engine — it comes with an API key "
                             "(create one on the app's /developer page, then suresuite.install_engine())")
        m = re.fullmatch(r"/runs/([^/:]+)(/replications|/validation|:cancel|:add-reps)?", p)
        if m:
            run = self.runs.get(m.group(1))
            if run is None:
                return self._err(404, "run_not_found", "run not found")
            return self._run(method, run, m.group(2) or "", params, body)
        return self._err(404, "route_not_found", f"no route for {method} {p}")

    def _project(self, method, sub, body, headers):
        if sub == "" and method == "GET":
            return self._ok(self.d["project"])
        if sub == "/dataset-versions" and method == "GET":
            return self._ok({"data": self.datasets})
        if sub == "/datasets:freeze" and method == "POST":
            dv = self.datasets[0]  # nothing changed since: the existing version comes back
            return self._ok({k: dv[k] for k in ("id", "label", "graph_hash", "created_at")}, 201)
        if sub == "/policy-catalog" and method == "GET":
            return self._ok(self.d["catalog"])
        if sub == "/policies" and method == "GET":
            return self._ok({"defaults": {"project_id": self.pid, **copy.deepcopy(self.defaults),
                                          "fulfillment_strategy": None, "active_preset": None,
                                          "created_at": self.TS, "updated_at": self.TS},
                             "overrides": copy.deepcopy(self.overrides)})
        if sub == "/policies" and method == "PUT":
            bad = set(body) - {"defaults", "fulfillment_strategy", "overrides"}
            bad |= set((body.get("defaults") or {})) - set(FAMILIES)
            if bad:
                return self._err(400, "invalid_request", f"unrecognized key(s): {sorted(bad)}")
            for fam, val in (body.get("defaults") or {}).items():
                self.defaults[fam] = copy.deepcopy(val)
            for o in body.get("overrides") or []:
                self.overrides = [x for x in self.overrides if (x["scope"], x["target_key"], x["family"])
                                  != (o["scope"], o["target_key"], o["family"])] + [{**o, "updated_at": self.TS}]
            return self._ok({"ok": True})
        if sub == "/policy-versions" and method == "POST":
            h = _sha({f: v for f, v in self.defaults.items() if v})
            for v in self.versions:  # content-addressed: the oldest holder of this config comes back
                if v["policy_hash"] == h:
                    return self._ok({k: v[k] for k in ("id", "label", "policy_hash", "created_at")}, 201)
            v = {"id": self._id("policy-version", len(self.versions) + 1), "label": body.get("label"),
                 "notes": None, "author_email": "demo@example.com", "author_name": "Demo",
                 "parent_version_id": None, "policy_hash": h, "created_at": self.TS, "run_count": 0,
                 "card_count": 0, "version_no": len(self.versions) + 1, "_policies": copy.deepcopy(self.defaults)}
            self.versions.append(v)
            return self._ok({k: v[k] for k in ("id", "label", "policy_hash", "created_at")}, 201)
        m = re.fullmatch(r"/dataset-versions/([^/]+)", sub)
        if m and method == "GET":
            dv = self.datasets[0] if m.group(1) in ("latest", self.datasets[0]["id"]) else None
            if dv is None:
                return self._err(404, "dataset_version_not_found", "no such dataset version in this project")
            snap = {"schema_version": 3, "inputs": {**copy.deepcopy(self.d["dataset"]), "customers": [],
                                                    "bom_multi_level": []}, "network": {}}
            return self._ok({**{k: dv[k] for k in ("id", "label", "version_no", "graph_hash", "author_email",
                                                   "created_at")},
                             "hash_inputs": None, "hash_network": None, "schema_version": 3, "snapshot": snap})
        m = re.fullmatch(r"/policy-versions/([^/]+)", sub)
        if m and method == "GET":
            vid = m.group(1)
            v = (self.versions[-1] if self.versions else None) if vid == "latest" else \
                next((x for x in self.versions if x["id"] == vid), None)
            if v is None:
                return self._err(404, "policy_version_not_found", "no such policy version in this project")
            return self._ok({**{k: x for k, x in v.items() if not k.startswith("_")},
                             "snapshot": {"schema_version": 2, "defaults": copy.deepcopy(v["_policies"]),
                                          "fulfillment_strategy": None, "overrides": []}})
        if sub == "/policy-versions" and method == "GET":
            return self._ok({"data": [{k: v for k, v in x.items() if not k.startswith("_")}
                                      for x in reversed(self.versions)]})
        if sub == "/scenarios" and method == "GET":
            return self._ok({"data": copy.deepcopy(self.scenarios), "next_cursor": None})
        if sub == "/scenarios" and method == "POST":
            try:
                sc = self._new_scenario(body)
            except ValueError as e:
                return self._err(400, "invalid_request", str(e))
            return self._ok({k: sc[k] for k in ("id", "name", "horizon_days", "replications", "seed", "created_at")}, 201)
        if sub == "/runs" and method == "POST":
            return self._dispatch(body, headers.get("Idempotency-Key"))
        return self._err(404, "route_not_found", f"no route for {method} /projects/…{sub}")

    def _dispatch(self, body, idem):
        bad = set(body) - set(RUN_BODY_FIELDS)
        if bad:
            return self._err(400, "invalid_request", f"unrecognized key(s): {sorted(bad)}")
        if idem and idem in self.idem:
            run = self.runs[self.idem[idem]]
            return 202, {"Idempotency-Replayed": "true"}, {"run_id": run["id"], "status": run["status"],
                                                          "policy_hash": run["policy_hash"], "graph_hash": run["graph_hash"]}
        sc = next((s for s in self.scenarios if s["id"] == body.get("scenario_id")), None)
        ver = next((v for v in self.versions if v["id"] == body.get("policy_version_id")), None)
        if sc is None or ver is None:
            return self._err(500, "dispatch_failed", "scenario or policy version does not belong to this project")
        findings = []
        for ev in sc["disruption_schedule"][:5]:
            raw = str(ev.get("target", ""))
            tid = raw.rsplit(":", 1)[-1]
            if tid in self.supplier_ids or raw.lower().startswith("plant:") or tid.lower() == "plant":
                continue
            kind = raw.split(":", 1)[0].lower() if ":" in raw else ""
            why = (f"The engine cannot disrupt {kind} targets yet — this event is skipped."
                   if kind in ("material", "edge", "customer", "lane")
                   else f"No supplier or plant named \"{tid}\" in this project's data — this event is skipped.")
            findings.append({"severity": "warn", "field": "scenarios.disruption_schedule", "policy": "engine",
                             "rows": [raw], "message": f"Disruption on \"{raw}\": {why}"})
        if findings and not body.get("acknowledge_warnings"):
            return self._err(422, "validation_failed", "the pre-run gate needs acknowledgement",
                             {"validation": "ack_required", "ack_required": True, "findings": findings})
        live_schedule = [ev for ev in sc["disruption_schedule"][:5]
                         if not any(f["rows"] == [str(ev.get("target", ""))] for f in findings)]
        key = _demo_key(ver["_policies"], live_schedule, sc)
        rec = self.recordings.get(key)
        if rec is None:
            labels = ", ".join(r["label"] for r in self.d["recordings"])
            return self._err(400, "demo_no_recording",
                             "the offline demo has no recording for this policy + scenario combination "
                             f"(recorded: {labels}). Use the notebook's suggested settings, or go live.")
        if not body.get("force_rerun"):
            for prev in self.runs.values():
                if prev["_key"] == key and prev["policy_version_id"] == ver["id"] and prev["status"] == "done":
                    return self._err(409, "reuse_available", "an identical completed run exists",
                                     {"reuse_candidate": {"run_id": prev["id"], "ended_at": prev["ended_at"],
                                                          "created_at": prev["created_at"],
                                                          "code_version": prev["code_version"],
                                                          "rep_count_done": prev["rep_count_done"]}})
        self._n += 1
        run = {f: None for f in RUN_FIELDS}
        run.update(id=self._id("run", self._n), scenario_id=sc["id"], project_id=self.pid, status="queued",
                   created_at=self.TS, updated_at=self.TS, rep_count_target=sc["replications"], rep_count_done=0,
                   policy_version_id=ver["id"], policy_hash=ver["policy_hash"],
                   dataset_version_id=self.datasets[0]["id"], graph_hash=self.d["graph_hash"],
                   scenario_hash=_sha({k: sc[k] for k in ("horizon_days", "warmup_days", "replications", "seed", "crn")}),
                   gate_skipped=False, _key=key, _polls=0)
        self.runs[run["id"]] = run
        if idem:
            self.idem[idem] = run["id"]
        return 202, {"X-Request-Id": "demo"}, {"run_id": run["id"], "status": "queued", "policy_hash": run["policy_hash"],
                                                "graph_hash": run["graph_hash"], "gate_skipped": False}

    def _advance(self, run):
        if run["status"] in TERMINAL:
            return
        run["_polls"] += 1
        rec = self.recordings[run["_key"]]
        if run["_polls"] == 1:
            run.update(status="running", started_at=self.TS, rep_count_done=run["rep_count_target"] // 2)
        else:
            res = rec["run"]
            run.update(status="done", ended_at=self.TS, rep_count_done=res["rep_count_done"],
                       aggregate_kpis=copy.deepcopy(res["aggregate_kpis"]),
                       ci_half_widths=copy.deepcopy(res["ci_half_widths"]),
                       warmup_detected_at=res.get("warmup_detected_at"), code_version=res["code_version"])

    def _run(self, method, run, sub, params, body):
        if sub == "" and method == "GET":
            self._advance(run)
            return self._ok({k: copy.deepcopy(run[k]) for k in RUN_FIELDS})
        if sub == "/replications" and method == "GET":
            reps = self.recordings[run["_key"]]["replications"] if run["status"] == "done" else []
            after = int(params.get("cursor")) if params.get("cursor") not in (None, "") else -1
            limit = max(1, min(100, int(params.get("limit") or 20)))
            rows = [copy.deepcopy(r) for r in reps if r["rep_index"] > after]
            page = rows[:limit]
            body = {"data": page, "next_cursor": str(page[-1]["rep_index"]) if len(rows) > limit else None}
            if params.get("include") == "time_series":
                for r in page:
                    r["time_series"] = {}
                body["series"] = {"location": "object", "format": "parquet (zstd), long form",
                                  "url": f"demo://series/{run['id']}", "expires_in_seconds": 600}
            return self._ok(body)
        if sub == "/validation" and method == "GET":
            return self._ok({"status": "unvalidated", "model_validation": None, "gate_skipped": False})
        if sub == ":cancel" and method == "POST":
            if run["status"] not in TERMINAL:
                run.update(status="cancelled", ended_at=self.TS)
            # Like the live API, the answer says "cancelled" even for a run that had
            # already finished — read the run back to see what really happened.
            return self._ok({"run_id": run["id"], "status": "cancelled"}, 202)
        if sub == ":add-reps" and method == "POST":
            return self._ok({"run_id": run["id"], "added": (body or {}).get("n")}, 202)
        return self._err(404, "route_not_found", f"no route for {method} /runs/…{sub}")


def _demo(demo_data):
    if isinstance(demo_data, str):  # the notebook embeds it compressed (raw deflate + base64)
        demo_data = json.loads(zlib.decompress(base64.b64decode(demo_data), -15))
    return SuReSuite(DemoTransport(demo_data), live=False)


def connect(base_url, api_key=None, demo_data=None, needs=(), mode="auto"):
    """A live client when you have a key, otherwise the offline demo.

    `needs` are the scopes the caller will use. A live key is checked against them
    BEFORE anything runs: one the platform would refuse partway through stops here
    and says what is missing — or, in `mode="auto"` with demo data at hand, says so
    and continues as the offline demo, so a notebook never breaks halfway."""
    if not api_key:
        return _demo(demo_data)
    api = SuReSuite(HttpTransport(base_url, api_key), live=True)
    if not needs:
        return api
    info = api.key()
    if info is None:  # a gateway without GET /key: nothing to check against
        return api
    missing = [s for s in needs if s not in (info.get("scopes") or [])]
    if not missing:
        return api
    why = (f"this key ({info.get('principal')} key sk_{info.get('env')}_{info.get('key_prefix')}…) cannot use "
           f"{', '.join(missing)}, which this notebook needs. Create a key on the app's /developer page with "
           f"{', '.join(needs)}, then set SURESUITE_API_KEY to it and run all cells again.")
    if info.get("principal") == "personal" and not any(s.startswith("write:") for s in info.get("scopes") or []):
        why += (" A personal key is read-only unless its owner is an organization admin or a modeler — "
                "ask one of them for the role or for a key.")
    if mode == "live" or demo_data is None:
        raise Stop(why)
    print(f"{why}\n\nUntil then this notebook runs in DEMO mode on the example project, so every cell "
          "still works.")
    return _demo(demo_data)


# GENERATED by scripts/notebooks/build-notebooks.mjs from src/lib/sim/kpiDisplay.ts — do not edit.
# The Simulation Lab's KPI vocabulary: key → (label, format, higher_is_better).
# higher_is_better is None where "better" is not defined (inventory, utilization).
KPI_DISPLAY = {
    "fill_rate": ("Fill rate (α)", "pct2", True),
    "revenue": ("Revenue", "money", True),
    "demand_value": ("Demand (value)", "money", None),
    "produced_value": ("Produced (value)", "money", None),
    "lost_sales_value": ("Lost sales (value)", "money", False),
    "lost_units": ("Lost units", "units", False),
    "lost_inbound_units": ("Lost inbound units", "units", False),
    "max_backlog": ("Peak backlog (units)", "units", False),
    "avg_on_hand_value": ("Average material inventory (value)", "money", None),
    "avg_fg_value": ("Average finished-goods inventory (value)", "money", None),
    "avg_on_hand_units": ("Average material inventory (units)", "units", None),
    "avg_fg_units": ("Average finished-goods inventory (units)", "units", None),
    "capacity_utilization": ("Plant capacity utilization", "pct1", None),
    "supplier_capacity_utilization": ("Supplier capacity utilization", "pct1", None),
    "products_capacity_bound": ("Products capacity held back", "units", False),
    "suppliers_capacity_bound": ("Suppliers capacity held back", "units", False),
    "cost_of_resilience": ("Cost of resilience", "money", False),
    "cost_ss_holding": ("Cost · safety-stock holding", "money", False),
    "cost_backup_premium": ("Cost · backup supplier premium", "money", False),
    "cost_multi_sourcing_premium": ("Cost · multi-sourcing premium", "money", False),
    "cost_expediting": ("Cost · expediting freight", "money", False),
    "cost_overtime": ("Cost · overtime capacity", "money", False),
    "cost_lost_sales": ("Cost · lost sales", "money", False),
    "cost_allocation_labor": ("Cost · allocation labour", "money", False),
    "cost_fg_ss_holding": ("Cost · finished-goods safety stock", "money", False),
    "cost_backorder_penalty": ("Cost · backorder penalty", "money", False),
    "cost_monitoring": ("Cost · supplier monitoring", "money", False),
    "ttr_weeks": ("Time to recover (weeks)", "fixed1", False),
    "tts_weeks": ("Time to survive (weeks)", "fixed1", True),
    "pre_disruption_fill_rate": ("Fill rate before the disruption", "pct2", True),
    "fill_rate_beta": ("Fill rate (β)", "pct2", True),
    "otif": ("OTIF", "pct2", True),
    "lead_time_days": ("Lead time (days)", "fixed2", False),
    "lead_time_p95": ("Lead time p95", "fixed2", False),
    "cost": ("Cost", "money", False),
    "profit": ("Profit", "money", True),
    "utilization": ("Utilization (avg) · legacy", "pct1", True),
    "inventory_turns": ("Inventory turns", "fixed2", True),
    "backorder_days": ("Backorder days", "fixed1", False),
    "ttr_days": ("Time-to-recover (days) · legacy", "fixed1", False),
}

In [ ]:
# @title Local simulation (the suresuite package's local.py)
"""Simulate on your own machine — Phase 12 · WP 12.5.

Pull the frozen inputs a platform run reads, then run the SAME engine function
the platform's browser engine and the Fly worker use
(`sim_worker.local.run_from_snapshots`), on your own CPU:

    import suresuite as ss
    api = ss.connect()                                  # key from SURESUITE_API_KEY
    ss.install_engine(api)                              # once per environment
    data = ss.dataset(api, project_id)                  # the latest frozen dataset version
    policy = ss.policy(api, project_id)                 # the latest policy version
    run = ss.simulate(data, policy, replications=50, seed=42,
                      disruptions=[ss.outage("S2", start_day=140, duration_days=42)])
    ss.kpi_table(run["run"])

`simulate` returns the shapes the API returns — a run row, replication rows and
a long-form weekly series — so every helper (kpi_table, paired_compare,
plot_series, …) works on a local run exactly as on a platform run.
"""
from __future__ import annotations

import hashlib
import importlib.util
import subprocess
import sys
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

# The Simulation Lab's defaults: 52 weeks (the engine's minimum), 10 replications,
# seed 42, common random numbers on, warm-up detected by the engine.
DEFAULT_FRAME = {"horizon_days": 364, "replications": 10, "seed": 42, "crn": True, "warmup_mode": "auto"}


# ── the engine ──────────────────────────────────────────────────────────────
def engine_available() -> bool:
    """Is the simulation engine importable in this environment?"""
    return importlib.util.find_spec("scsim") is not None and importlib.util.find_spec("sim_worker") is not None


def install_engine(api, quiet: bool = True, version: Optional[str] = None) -> str:
    """Download the engine through the API (a key with read:data), check every
    wheel's sha256, and pip-install it. Returns the engine version.

    ``version`` installs an EARLIER engine — ``"0.4.0"`` (the newest build of that
    version) or the exact build a run recorded, ``run["code_version"]``
    (``"scsim-0.4.0+<digest>"``) — so a stored result can be re-run on the engine
    that produced it. Omit it for the engine the platform runs now."""
    info = api.engine(version=version) if version else api.engine()
    with tempfile.TemporaryDirectory() as tmp:
        paths = []
        for w in info["wheels"]:
            body = requests.get(w["url"], timeout=120).content  # a signed URL: no API key sent
            digest = hashlib.sha256(body).hexdigest()
            if digest != w["sha256"]:
                raise RuntimeError(f"{w['file']}: sha256 {digest[:12]}… is not the published {w['sha256'][:12]}… — "
                                   "not installing it")
            path = Path(tmp) / w["file"]
            path.write_bytes(body)
            paths.append(str(path))
        cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *(["-q"] if quiet else []), *paths]
        subprocess.check_call(cmd)
    importlib.invalidate_caches()
    build = info.get("engine_build") or "build not named"
    print(f"engine {info['engine_version']} ({build}) installed ({len(paths)} wheels, sha256 checked)")
    return info["engine_version"]


def _require_engine():
    if not engine_available():
        raise ImportError("the simulation engine is not installed here — run suresuite.install_engine(api) "
                          "(it needs an API key with read:data)")


# ── the frozen inputs ───────────────────────────────────────────────────────
@dataclass
class Dataset:
    """A frozen dataset version: its identity and its input tables."""
    id: str
    graph_hash: Optional[str]
    snapshot: dict
    label: Optional[str] = None
    version_no: Optional[int] = None
    meta: dict = field(default_factory=dict)

    @property
    def tables(self) -> dict[str, pd.DataFrame]:
        """The simulation's input tables, one DataFrame each."""
        inputs = self.snapshot.get("inputs") if isinstance(self.snapshot.get("inputs"), dict) else self.snapshot
        return {name: pd.DataFrame(rows) for name, rows in inputs.items() if isinstance(rows, list)}

    def __repr__(self) -> str:
        sizes = ", ".join(f"{k} {len(v)}" for k, v in self.tables.items() if len(v))
        return f"Dataset(v{self.version_no} {self.label!r}, graph_hash {str(self.graph_hash)[:12]}…: {sizes})"


@dataclass
class Policy:
    """A frozen policy version: its identity and its snapshot."""
    id: str
    policy_hash: Optional[str]
    snapshot: dict
    label: Optional[str] = None


def dataset(api, project_id: str, version: str = "latest") -> Dataset:
    """Pull a frozen dataset version (id or "latest") with its rows."""
    d = api.dataset_version(project_id, version)
    return Dataset(id=d["id"], graph_hash=d.get("graph_hash"), snapshot=d["snapshot"], label=d.get("label"),
                   version_no=d.get("version_no"), meta={k: v for k, v in d.items() if k != "snapshot"})


def policy(api, project_id: str, version: str = "latest") -> Policy:
    """Pull a frozen policy version (id or "latest")."""
    p = api.policy_version(project_id, version)
    return Policy(id=p["id"], policy_hash=p.get("policy_hash"), snapshot=p["snapshot"], label=p.get("label"))


def with_tables(data: Dataset, **tables: pd.DataFrame | list) -> Dataset:
    """A copy of `data` with some input tables replaced — a what-if on your own
    machine (more capacity, a second supplier, …). The platform's data is untouched."""
    snap = dict(data.snapshot)
    inputs = dict(snap.get("inputs") or {})
    for name, rows in tables.items():
        inputs[name] = rows.to_dict("records") if isinstance(rows, pd.DataFrame) else list(rows)
    snap["inputs"] = inputs
    return Dataset(id=f"{data.id} (edited locally)", graph_hash=None, snapshot=snap, label=f"{data.label} + local edits",
                   version_no=data.version_no, meta=dict(data.meta))


# ── the run ─────────────────────────────────────────────────────────────────
def simulate(data: Dataset | dict, policy_version: Policy | dict, scenario: Optional[dict] = None,
             disruptions: Optional[list] = None, project_model: Optional[str] = None,
             progress: bool = False, **frame: Any) -> dict:
    """Run the platform's engine on this machine. `scenario` is a scenario row (from
    `api.scenarios(...)`) or None; keyword arguments override its frame
    (horizon_days, replications, seed, crn, warmup_mode, …). Returns
    {"run", "replications", "series", "inputs"} in the API's shapes."""
    _require_engine()
    from sim_worker.local import run_from_snapshots

    snap = data.snapshot if isinstance(data, Dataset) else data
    pol = policy_version.snapshot if isinstance(policy_version, Policy) else policy_version
    sc = {**DEFAULT_FRAME, **{k: v for k, v in (scenario or {}).items()
                             if k in ("horizon_days", "warmup_days", "warmup_mode", "replications", "seed", "crn",
                                      "disruption_schedule", "stopping_rule", "demand_model", "recovery_overrides")},
          **frame}
    if disruptions is not None:
        sc["disruption_schedule"] = disruptions

    def _tick(_row, done, total):
        print(f"  replication {done}/{total}   ", end="\r")

    out = run_from_snapshots(snap, pol, sc, project_model=project_model, on_replication=_tick if progress else None)
    run = {**out["run_update"], "status": "done", "computed_by": "client", "engine_version": out["engine_version"],
           "graph_hash": getattr(data, "graph_hash", None), "policy_hash": getattr(policy_version, "policy_hash", None),
           "dataset_version_id": getattr(data, "id", None), "policy_version_id": getattr(policy_version, "id", None),
           "rep_count_target": sc["replications"]}
    if progress:
        print()
    return {"run": run, "replications": out["replications"], "series": pd.DataFrame(out["series"]),
            "inputs": {"scenario": sc}}

In [ ]:
# @title Study helpers (shared by notebooks 04 and 05)
# ── SuReSuite study helpers ─────────────────────────────────────────────────
# Shared by notebooks 04 and 05, so both read your project the same way: the data a
# run reads (frozen, or the newest frozen copy and its age), the policies as they are
# on /policies now (saved, or rebuilt here for a read-only key), your saved baseline
# scenario's frame, the platform's engine, and one replication's per-item weeks.
# Run this cell; you do not need to read it.
import json
import warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

INPUT_TABLES = ("suppliers", "materials", "products", "customers", "inbound", "outbound",
                "bom", "bom_multi_level", "demand_forecasts")


def key_scopes(api):
    """The scopes this key may use now (GET /key), so each step knows its fallback."""
    return set((api.key() or {}).get("scopes") or ["read:data", "read:policies"])


def key_table(scopes, verify_use="VERIFY_ON_PLATFORM"):
    return pd.DataFrame([
        {"needs": s, "this key": "yes" if s in scopes else "no", "used for": use, "without it": alt}
        for s, use, alt in [
            ("read:data", "your data and the engine", "—"),
            ("read:policies", "your policies", "—"),
            ("read:runs", "your saved baseline and stress-test scenarios", "the Simulation Lab's defaults"),
            ("write:data", "freezing the data as it is now", "the newest frozen copy (its age is shown)"),
            ("write:policies", "saving /policies as a version", "/policies read live and rebuilt here"),
            ("write:runs", verify_use, "skipped"),
        ]]).set_index("needs")


def styled(df, fmt=None, shade=(), cmap="Reds", caption=None):
    """A formatted table with value shading, or the plain table where the notebook cannot style."""
    try:
        s = df.style.format(fmt or {}, na_rep="—")
        for c in shade:
            if c in df and pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any():
                s = s.background_gradient(cmap=cmap, subset=[c])
        return s.set_caption(caption) if caption else s
    except (ImportError, AttributeError):  # pandas styling needs jinja2; without it `.style` raises AttributeError
        if caption:
            print(caption)
        return df


def capped(df, what="rows", limit=None):
    limit = limit or globals().get("MAX_ROWS", 40)
    if len(df) > limit:
        print(f"showing {limit} of {len(df)} {what} — the export has all of them")
        return df.head(limit)
    return df


def col(df, name, default=None):
    return df[name] if name in df else pd.Series(default, index=df.index)


def load_dataset(api, project_id, source, scopes, live, label):
    """The frozen dataset version a run reads. "current" freezes the data now when the key may
    (an unchanged project returns the version it has); without write:data it is the newest
    version, and its age is said. Versions are immutable, so they are cached on disk by id."""
    versions = sorted(api.dataset_versions(project_id), key=lambda v: v.get("created_at") or "", reverse=True)
    if source == "current" and "write:data" in scopes:
        version_id = api.freeze_dataset(project_id, label)["id"]
        print("froze the data as it is now:", "no change since the last freeze, so that version is used"
              if version_id in {v["id"] for v in versions} else "a new version")
    elif source in ("current", "latest"):
        if not versions:
            raise Stop("this project has no frozen data yet — run anything in the Simulation Lab (every run "
                       "freezes the data it reads) or use a key with write:data, then run again")
        version_id = versions[0]["id"]
        if source == "current":
            age = (pd.Timestamp.now(tz="UTC") - pd.Timestamp(versions[0]["created_at"])).days
            print(f"note: this key cannot freeze data (no write:data), so the newest frozen version is used — "
                  f"it is {age} day(s) old. Anything uploaded since is NOT in it. Every platform run freezes "
                  f"the data it reads: run anything in the Simulation Lab to refresh it.")
    else:
        version_id = source
    path = Path("suresuite_cache") / project_id / f"{version_id}.json"
    started = time.time()
    if path.exists():
        dv = json.loads(path.read_text())
        print(f"dataset version {version_id}: from the local cache")
    else:
        dv = api.dataset_version(project_id, version_id)
        if live:  # a version never changes, so a copy on disk is always current
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(dv))
        print(f"dataset version {version_id}: downloaded {len(json.dumps(dv)) / 1e6:.1f} MB "
              f"in {time.time() - started:.1f}s")
    return Dataset(id=dv["id"], graph_hash=dv.get("graph_hash"), snapshot=dv["snapshot"], label=dv.get("label"),
                   version_no=dv.get("version_no"), meta={k: v for k, v in dv.items() if k != "snapshot"})


def input_tables(data):
    return {name: data.tables.get(name, pd.DataFrame()) for name in INPUT_TABLES}


def flat_bom(T):
    """(product, material, rate): units of each purchased material per unit of each product,
    through any sub-assemblies. The multi-level BOM wins when it has rows, as in the engine."""
    ml = T["bom_multi_level"]
    if len(ml):
        children = {}
        for parent, child, rate in zip(col(ml, "higher_level_component_id"), ml["material_id"],
                                       col(ml, "consumption_rate", 1)):
            if parent:
                children.setdefault(str(parent), []).append((str(child), float(rate or 0)))
        out = {}

        def walk(product, node, mult, depth=0):
            for child, rate in children.get(node, []):
                if child in children and depth < 25:
                    walk(product, child, mult * rate, depth + 1)
                else:
                    out[(product, child)] = out.get((product, child), 0.0) + mult * rate
        for p in col(T["products"], "product_id").astype(str):
            walk(p, p, 1.0)
        return pd.DataFrame([{"product_id": p, "material_id": m, "rate": r} for (p, m), r in out.items()],
                            columns=["product_id", "material_id", "rate"])
    b = T["bom"]
    return pd.DataFrame({"product_id": col(b, "product_id").astype(str), "material_id": col(b, "material_id").astype(str),
                         "rate": col(b, "consumption_rate", 1).astype(float)})


PER_WEEK = {"day": 7, "week": 1, "month": 12 / 52, "quarter": 4 / 52, "year": 1 / 52}


def supply_lanes(T):
    """Inbound lanes with their weekly volume and weekly spend (for ranking only; the engine
    normalizes the data itself)."""
    return T["inbound"].assign(
        supplier_id=lambda d: col(d, "supplier_id").astype(str), material_id=lambda d: col(d, "material_id").astype(str),
        per_week=lambda d: col(d, "volume", 0).astype(float).fillna(0)
                           * col(d, "time_unit", "week").fillna("week").str.lower().map(PER_WEEK).fillna(1),
        spend=lambda d: d["per_week"] * col(d, "unit_price", 0).astype(float).fillna(0))


def data_check(T, bom, lanes):
    mats = set(col(T["materials"], "material_id").astype(str))
    prods = set(col(T["products"], "product_id").astype(str))
    n_sources = lanes.groupby("material_id")["supplier_id"].nunique()
    return pd.DataFrame([
        ("materials in the BOM with no item master", sorted(set(bom.material_id) - mats)),
        ("materials with no supplier lane", sorted(set(bom.material_id) - set(lanes.material_id))),
        ("products with no BOM", sorted(prods - set(bom.product_id))),
        ("materials with a single supplier", sorted(n_sources[n_sources == 1].index)),
    ], columns=["check", "items"]).assign(
        count=lambda d: d["items"].map(len),
        items=lambda d: d["items"].map(lambda xs: ", ".join(xs[:12]) + (" …" if len(xs) > 12 else "")))


def load_policies(api, project_id, source, pinned_id, scopes, label):
    """The policy version a run reads, and what differs between the saved version and /policies now.
    Returns (policy, drift rows, the families as stored on /policies now)."""
    saved = sorted(api.policy_versions(project_id), key=lambda v: v.get("created_at") or "", reverse=True)
    pinned_id = pinned_id or (saved[0]["id"] if saved else None)
    pinned = policy(api, project_id, pinned_id) if pinned_id else None

    live = api.policies(project_id)
    live_def = live.get("defaults") or {}
    now_fam = {f: dict(live_def.get(f) or {}) for f in FAMILIES}
    now_strategy = live_def.get("fulfillment_strategy")
    canon = lambda o: (o["scope"], o["target_key"], o["family"])
    now_over = {canon(o): o.get("patch") or {} for o in live.get("overrides") or []}

    drift = []
    if pinned:
        saved_def = pinned.snapshot.get("defaults") or {}
        for f in FAMILIES:
            for k, v in now_fam[f].items():
                if (saved_def.get(f) or {}).get(k) != v:
                    drift.append({"setting": f"{f}.{k}", "saved version": (saved_def.get(f) or {}).get(k), "/policies now": v})
        if now_strategy and now_strategy != pinned.snapshot.get("fulfillment_strategy"):
            drift.append({"setting": "fulfillment_strategy", "saved version": pinned.snapshot.get("fulfillment_strategy"),
                          "/policies now": now_strategy})
        saved_over = {canon(o): o.get("patch") or {} for o in pinned.snapshot.get("overrides") or []}
        for key in sorted(set(saved_over) | set(now_over)):
            if saved_over.get(key) != now_over.get(key):
                drift.append({"setting": "override " + " · ".join(key), "saved version": saved_over.get(key),
                              "/policies now": now_over.get(key)})
        print(f"saved version: {pinned.label or 'unlabelled'} ({pinned.id})")

    if source == "pinned" or (pinned and not drift):
        if not pinned:
            raise Stop("POLICY_SOURCE is \"pinned\" but this project has no saved policy version")
        pol = pinned
        print("using the saved version" + ("" if not drift else f" — {len(drift)} setting(s) differ from /policies now (below)"))
    elif "write:policies" in scopes:
        pol = policy(api, project_id, api.snapshot_policies(project_id, f"{label} · /policies as it is now")["id"])
        print(f"saved /policies as it is now: {pol.label} ({pol.id})")
    else:
        # The platform builds a version as the page's defaults with the stored values over them, family
        # by family. The saved version already holds the page's defaults, so the stored values go over it.
        snap = copy.deepcopy(pinned.snapshot) if pinned else {"schema_version": 2, "defaults": {}}
        for f in FAMILIES:
            snap.setdefault("defaults", {})[f] = {**(snap["defaults"].get(f) or {}), **now_fam[f]}
        snap["fulfillment_strategy"] = now_strategy or snap.get("fulfillment_strategy") or "make_to_stock"
        snap["overrides"] = [{"scope": s, "target_key": t, "family": f, "patch": p} for (s, t, f), p in sorted(now_over.items())]
        pol = Policy(id="rebuilt from /policies (not a saved version)", policy_hash=None, snapshot=snap,
                     label="/policies as it is now")
        print("note: this key cannot save a policy version (no write:policies), so the current policies are rebuilt "
              "here from /policies" + (" over the saved version." if pinned else
              " alone — there is no saved version, so settings never touched on /policies take the ENGINE's defaults, "
              "which can differ from the page's. Save a version on /policies for an exact run."))
    if not drift and pinned:
        print("the saved version matches /policies on every stored setting")
    return pol, pd.DataFrame(drift, columns=["setting", "saved version", "/policies now"]).set_index("setting"), now_fam


def policy_tables(pol, now_fam):
    """(project-wide settings with where each comes from, per-item overrides counted by family)."""
    snap = pol.snapshot or {}
    rows = [{"family": f, "setting": k, "value": v if not isinstance(v, (dict, list)) else json.dumps(v)[:80],
             "where": "set on /policies" if k in now_fam.get(f, {}) else "page default"}
            for f in FAMILIES for k, v in (snap.get("defaults", {}).get(f) or {}).items()]
    settings = pd.DataFrame(rows, columns=["family", "setting", "value", "where"]).sort_values(
        ["where", "family"], ascending=[False, True])
    over = pd.DataFrame(snap.get("overrides") or [], columns=["scope", "target_key", "family", "patch"])
    return settings, over.groupby(["family", "scope"]).size().rename("per-item overrides").to_frame()


def study_scenario(api, project_id, scenario_id, scopes, replications=None):
    """Your saved baseline scenario (or the Simulation Lab's defaults), its frame, and your saved
    stress tests. A pre-filled SCENARIO_ID with disruptions is used as a frame and its events
    join the stress tests."""
    scenarios = api.scenarios(project_id) if "read:runs" in scopes else []
    stress = [s for s in scenarios if s.get("disruption_schedule")]
    sc = next((s for s in scenarios if s["id"] == scenario_id), None) if scenario_id else \
        next((s for s in scenarios if not s.get("disruption_schedule")), None)
    if sc is None:
        sc = {"id": None, "name": "Simulation Lab defaults", **DEFAULT_FRAME}
        print("no saved baseline scenario" + ("" if "read:runs" in scopes else " (the key has no read:runs)")
              + " — using the Simulation Lab's defaults")
    elif sc.get("disruption_schedule"):
        print(f"note: {sc['name']} has disruptions — its frame is the baseline here, its events are a stress test")
        stress = [sc] + [s for s in stress if s["id"] != sc["id"]]
        sc = {**sc, "disruption_schedule": []}
    frame = {k: sc[k] for k in ("horizon_days", "warmup_days", "replications", "seed", "crn", "recovery_overrides")
             if sc.get(k) is not None}
    if replications:
        frame["replications"] = replications
    print(f"scenario: {sc['name']} — {frame.get('horizon_days')} days, {frame.get('replications')} replications, "
          f"seed {frame.get('seed')}")
    return sc, frame, stress


def ensure_engine(api, live):
    """The platform's current engine on this machine: installed, or reinstalled when its version
    is not the platform's. Demo mode hands out no engine."""
    try:
        platform_engine = api.engine()
    except SuReSuiteError:
        platform_engine = None
    if platform_engine and live:
        import importlib.metadata as md
        have = md.version("scsim") if engine_available() else None
        if have != platform_engine["engine_version"]:
            install_engine(api)
    if not engine_available():
        print("the engine is not installed — demo mode cannot hand it out. Add SURESUITE_API_KEY (see the Setup "
              "notes) and run all cells again; the simulation steps need it.")
        return False
    return True


def engine_view(data, pol, scenario, project_model):
    """What the engine RECEIVES for these inputs — lead times in weeks, capacities, opening stock,
    the allocation rule — read from the engine's own mapping rather than re-derived here."""
    from sim_worker.local import engine_input_from_snapshots
    return engine_input_from_snapshots(data.snapshot, pol.snapshot, scenario, project_model=project_model)["scenario"]


def inspect_run(data, pol, scenario, project_model, seed, disruptions=None):
    """One replication with every item's weekly record — {(kind, measure): weeks × items}. A one-replication
    run with seed s is replication 0 of any run with root seed s, so it is one of that run's own worlds."""
    from sim_worker.local import run_from_snapshots
    out = run_from_snapshots(data.snapshot, pol.snapshot,
                             {**scenario, "replications": 1, "seed": seed, "inspection": True,
                              "disruption_schedule": list(disruptions or [])},
                             project_model=project_model)
    frames = {}
    for r in out["item_series"]:
        for measure, values in r["series"].items():
            frames.setdefault((r["kind"], measure), {})[str(r["item_id"])] = values
    if not frames:
        raise Stop("the engine returned no per-item weeks — this engine build predates them; install the platform's")
    return out["run_update"], {k: pd.DataFrame(v) for k, v in frames.items()}


def use_of(frames, bom):
    """Each material used per week: each product's production times its BOM."""
    prod = frames.get(("product", "production"), pd.DataFrame())
    oh = frames[("material", "on_hand")]
    use = pd.DataFrame(0.0, index=oh.index, columns=oh.columns)
    for p, m, rate in bom.itertuples(index=False):
        if p in prod and m in use:
            use[m] += prod[p] * rate
    return use

In [ ]:
# @title Demo data (recorded engine output)
# ── DEMO DATA ───────────────────────────────────────────────────────────────
# Recorded engine output (scsim 0.8.0) for the Example project, replayed
# when no API key is found. Recorded by notebooks/tools/record_demo.py; not used in live mode.
# Recordings: baseline.
DEMO_DATA = "..."

## Try changing
- `REQUEST_KIND` — `"product"`: a customer's make-to-order request; `"material"`: material taken from stock.
- `ITEM_ID`, `QUANTITY`, `NEEDED_BY` (a date) or `NEEDED_WEEK` — what, how many, by when.
- `CONFIDENCE` — promise only what holds in at least this share of futures; `N_WORLDS` — how many futures.
- `SEARCH_WEEKS` — how far each way to look for a pull-in or a push-back.
- `DELIVERY_WEEKS` — transit to the customer, which the engine does not model; `None` reads your customer lanes.
- `PLAN_MATERIALS` — answer with your replenishment as it is, with the request's materials planned from the order book, or both.

In [ ]:
REQUEST_KIND = "product"     # "product" | "material"
ITEM_ID = ""                 # a product or material id; "" = the first product (or material) in your data
QUANTITY = 150
NEEDED_BY = ""               # "YYYY-MM-DD"; "" = NEEDED_WEEK
NEEDED_WEEK = 20             # weeks from week 0 (START_DATE)
START_DATE = ""              # the date of week 0; "" = your earliest forecast period, else today
DELIVERY_WEEKS = None        # product only: weeks in transit to the customer; None = the product's lanes
CONFIDENCE = 0.9
N_WORLDS = 10
SEARCH_WEEKS = 3
SERVE_LAST = True            # product only: True = after every existing customer row; False = your allocation rule
PLAN_MATERIALS = "both"      # product only: "as is" | "order book" (MRP for the request's materials) | "both"
DATA_SOURCE = "current"      # "current" | "latest" | a dataset version id
POLICY_SOURCE = "current"    # "current" | "pinned" (POLICY_VERSION_ID exactly)
MAX_ROWS = 40
EXPORT_XLSX = "available_to_promise.xlsx"   # "" = no export

## 1 · Connect, load the data and the policies
The same reading as notebook 04: the data a run reads (frozen now, or the newest frozen copy and its
age), the policies as they are on /policies now, and your saved baseline scenario's horizon.
**Expect:** your project, what the key allows, the data and policy versions used, and the scenario.

In [ ]:
import matplotlib.ticker as mticker
from IPython.display import Markdown

api = connect(BASE_URL, API_KEY, DEMO_DATA, needs=["read:data", "read:policies"], mode=MODE)
LIVE = api.live
project = pick_project(api, PROJECT_ID)
PID = project["id"]
PROJECT_MODEL = project.get("supply_chain_model")
SCOPES = key_scopes(api)
display(key_table(SCOPES, verify_use="(not used here)"))

data = load_dataset(api, PID, DATA_SOURCE, SCOPES, LIVE, "notebook 05")
print(data)
pol, drift, now_fam = load_policies(api, PID, POLICY_SOURCE, POLICY_VERSION_ID, SCOPES, "notebook 05")
if len(drift):
    display(styled(capped(drift, "differences"), caption="Saved version vs /policies now"))
scenario, FRAME, _ = study_scenario(api, PID, SCENARIO_ID, SCOPES)
RUN_LOCAL = ensure_engine(api, LIVE)
if not RUN_LOCAL:
    raise Stop("this notebook plays the request through the engine — add SURESUITE_API_KEY and run all cells again")

T = input_tables(data)
BOM = flat_bom(T)
lanes = supply_lanes(T)
FRAME_ALL = {**scenario, **FRAME}
EV = engine_view(data, pol, FRAME_ALL, PROJECT_MODEL)  # what the engine receives: lead times, capacities, stock
NET = EV["network"]
H = int(round(float(FRAME_ALL.get("horizon_days") or 364) / 7))

## 2 · The request, in the engine's weeks
Week 0 is the start of the simulation: your opening stock as it is in the data. Lead times and capacities
below are what the engine received, not a second reading of your tables.
**Expect:** the request, the week it falls in, and every lead time that stands between it and delivery.

In [ ]:
def to_date(week):
    return (START + pd.Timedelta(weeks=int(week))).date()


fc = T["demand_forecasts"]
START = pd.Timestamp(START_DATE) if START_DATE else (
    pd.to_datetime(col(fc, "period_start")).min() if len(fc) else pd.Timestamp.today().normalize())
if pd.isna(START):
    START = pd.Timestamp.today().normalize()
NEED_WEEK = int((pd.Timestamp(NEEDED_BY) - START).days // 7) if NEEDED_BY else int(NEEDED_WEEK)

products = {p["id"]: p for p in NET["products"]}
materials = {m["id"]: m for m in NET["materials"]}
links = pd.DataFrame(NET["supplier_links"])
suppliers = {s["id"]: s for s in NET["suppliers"]}
ITEM = ITEM_ID or (next(iter(products)) if REQUEST_KIND == "product" else next(iter(materials)))
if ITEM not in (products if REQUEST_KIND == "product" else materials):
    raise Stop(f"{ITEM!r} is not a {REQUEST_KIND} the engine received — check ITEM_ID against your data")


def lead_weeks(material):
    """The material's replenishment lead time in weeks: its primary supplier link's, else the shortest."""
    ls = links[links.material_id == material]
    if ls.empty:
        return None
    primary = ls[ls["primary"] == True]  # noqa: E712 — a column of booleans
    return int((primary if len(primary) else ls)["lead_time_weeks"].min())


if REQUEST_KIND == "product":
    if DELIVERY_WEEKS is not None:
        TRANSIT = int(DELIVERY_WEEKS)
    else:
        ob = T["outbound"]
        lt = pd.to_numeric(col(ob[col(ob, "product_id").astype(str) == ITEM], "expected_lead_time"), errors="coerce").dropna()
        TRANSIT = int(round(lt.median())) if len(lt) else 0
    SHIP_WEEK = NEED_WEEK - TRANSIT
    need = BOM[BOM.product_id == ITEM].assign(needed=lambda d: d.rate * QUANTITY)
    p = products[ITEM]
    facts = {
        "request": f"{QUANTITY:,.0f} × {ITEM} ({p.get('name') or ''})", "needed by": f"{to_date(NEED_WEEK)} (week {NEED_WEEK})",
        "transit to customer": f"{TRANSIT} wk → ship by week {SHIP_WEEK} ({to_date(SHIP_WEEK)})",
        "made": str(p.get("fulfillment_mode")).upper(), "plant capacity": f"{p.get('production_capacity'):,.0f} / wk",
        "production lead time": f"{p.get('production_lead_time_weeks') or 0:g} wk",
    }
    display(pd.Series(facts, name="").to_frame())
    MATS = list(need.material_id)
    display(styled(need.set_index("material_id").assign(
        **{"lead time (wk)": [lead_weeks(m) for m in need.material_id],
           "opening stock": [materials.get(m, {}).get("initial_on_hand") for m in need.material_id],
           "suppliers": [", ".join(sorted(links.supplier_id[links.material_id == m])) or "—" for m in need.material_id]}
    )[["rate", "needed", "opening stock", "lead time (wk)", "suppliers"]],
        fmt={"rate": "{:g}", "needed": "{:,.0f}", "opening stock": "{:,.0f}"},
        caption="What the request needs from each material"))
else:
    SHIP_WEEK = NEED_WEEK
    MATS = [ITEM]
    m = materials[ITEM]
    display(pd.Series({
        "request": f"{QUANTITY:,.0f} × {ITEM} ({m.get('name') or ''}) from stock",
        "needed by": f"{to_date(NEED_WEEK)} (week {NEED_WEEK})", "opening stock": f"{m.get('initial_on_hand') or 0:,.0f}",
        "replenishment lead time": f"{lead_weeks(ITEM)} wk",
        "suppliers": ", ".join(sorted(links.supplier_id[links.material_id == ITEM])) or "—",
    }, name="").to_frame())
if not 0 <= SHIP_WEEK < H:
    raise Stop(f"the request falls in week {SHIP_WEEK}, outside the simulated weeks 0–{H - 1}")
print("note: the data holds no open purchase orders, so the first lead-time weeks can understate supply you "
      "have already ordered.")

## 3 · Stock position: what you hold, what is coming, what is free
The project as it is, in `N_WORLDS` futures. **Available to promise** at a week is what you can take
out then without running short before a reorder can arrive: the lowest stock between that week and one
lead time later, at the `CONFIDENCE` level across futures.
**Expect:** a table per material at the needed week, and a chart of each material's stock and its ATP.

In [ ]:
SEEDS = [int(FRAME_ALL.get("seed", 42)) + i for i in range(N_WORLDS)]
started = time.time()
WORLDS = {s: inspect_run(data, pol, FRAME_ALL, PROJECT_MODEL, s)[1] for s in SEEDS}
ONE_RUN = (time.time() - started) / len(SEEDS)
print(f"{len(SEEDS)} futures in {time.time() - started:.1f}s")


def stock(frames_by_seed, material, measure="on_hand"):
    """weeks × seeds for one material."""
    return pd.DataFrame({s: f[("material", measure)][material] for s, f in frames_by_seed.items()})


def free_stock(oh, lead):
    """For each week, the lowest stock from that week to one lead time later — what can leave then."""
    return pd.DataFrame({w: oh.iloc[w:min(w + lead, len(oh) - 1) + 1].min() for w in oh.index}).T


q = 1 - CONFIDENCE
pos = []
for m in MATS:
    oh, lt = stock(WORLDS, m), lead_weeks(m) or 2
    atp = free_stock(oh, lt).quantile(q, axis=1)
    use = pd.DataFrame({s: use_of(f, BOM)[m] for s, f in WORLDS.items()})
    pos.append({"material": m, "opening stock": materials.get(m, {}).get("initial_on_hand"),
                f"on hand wk {SHIP_WEEK} (median)": oh.loc[SHIP_WEEK].median(),
                f"on hand wk {SHIP_WEEK} (low, {q:.0%})": oh.loc[SHIP_WEEK].quantile(q),
                "in transit (median)": stock(WORLDS, m, "in_transit").loc[SHIP_WEEK].median(),
                "use / wk": use.mean().mean(), "lead time (wk)": lt,
                f"ATP wk {SHIP_WEEK}": atp.loc[SHIP_WEEK]})
POSITION = pd.DataFrame(pos).set_index("material")
display(styled(POSITION, fmt={c: "{:,.0f}" for c in POSITION.columns if c != "lead time (wk)"},
               shade=[f"ATP wk {SHIP_WEEK}"], cmap="Greens",
               caption=f"Stock position at week {SHIP_WEEK} ({to_date(SHIP_WEEK)}) — ATP at {CONFIDENCE:.0%} confidence"))

for m in MATS[:4]:
    oh, lt = stock(WORLDS, m), lead_weeks(m) or 2
    atp = free_stock(oh, lt).quantile(q, axis=1)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.fill_between(oh.index, oh.quantile(0.1, axis=1), oh.quantile(0.9, axis=1), color="#2a78d6", alpha=0.15,
                    linewidth=0, label="on hand, 10th–90th percentile")
    ax.plot(oh.index, oh.median(axis=1), color="#2a78d6", linewidth=1.8, label="on hand, median")
    ax.step(atp.index, atp.values, where="post", color="#008300", linewidth=1.8, label=f"available to promise ({CONFIDENCE:.0%})")
    want = QUANTITY if REQUEST_KIND == "material" else float(need.set_index("material_id").loc[m, "needed"])
    ax.scatter([SHIP_WEEK], [want], color="#d9480f", zorder=5, s=40, label=f"request: {want:,.0f} in week {SHIP_WEEK}")
    ax.axvline(SHIP_WEEK, color="#d9480f", linewidth=0.8, linestyle=":")
    ax.set_title(f"{m} — lead time {lt} wk", fontsize=10)
    ax.set_xlabel("week")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.tight_layout()
    plt.show()

## 4 · Play the request through the chain
**A product:** the request becomes a firm order from a customer row added to a copy of your data
(`ATP-REQUEST`), in the ship week, waiting if it cannot be met at once. The same futures are run with the
row ordering nothing, so the only difference is the order. Every candidate week from `SEARCH_WEEKS` before
to `SEARCH_WEEKS` after is tried, which is what a pull-in or push-back would be. The request's deliveries
are the product's extra deliveries plus what existing customers of the product lost to it (served last,
they lose only to material or capacity the request used earlier).
**A material:** the engine has no way to take material out of stock directly, so the answer is read from
step 3's stock: the request fits a week when stock stays above it until a reorder can arrive.
**Expect:** the time it will take, then one row per candidate week.

In [ ]:
WEEKS = [w for w in range(SHIP_WEEK - SEARCH_WEEKS, SHIP_WEEK + SEARCH_WEEKS + 1) if 0 <= w < H]
TOL = max(0.5, 0.001 * QUANTITY)

def candidate_row(w, delivered, harm):
    """One candidate week from per-future results: delivered (weeks × futures, capped at the
    quantity) and harm (units existing customers lost, per future)."""
    done = delivered.ge(QUANTITY - TOL)
    when = done.apply(lambda c: c[c.index >= w].idxmax() if c[c.index >= w].any() else np.nan)
    return {"week": w, "date": to_date(w), "on time": (when <= w).mean(),
            f"on time at {CONFIDENCE:.0%}": delivered.loc[w].quantile(q),
            f"complete by (at {CONFIDENCE:.0%})": when.quantile(CONFIDENCE, interpolation="higher")
            if when.notna().mean() >= CONFIDENCE else np.nan,
            "others lose (median)": harm.median() if harm is not None else np.nan}


CANDS, TRIALS, ZEROS, SPLITS = {}, {}, {}, {}
if REQUEST_KIND == "product":
    REQ = "ATP-REQUEST"
    rule = (EV["policies"].get("customer_allocation") or {}).get("rule")
    ic = EV["policies"].get("inventory_control") or {}
    mrp_already = all(((ic.get("material_overrides") or {}).get(m) or {}).get("policy_type", ic.get("policy_type")) == "mrp"
                      for m in MATS)
    MODES = {"as is": False, "order book": True}
    MODES = {k: v for k, v in MODES.items() if PLAN_MATERIALS in (k, "both")}
    if mrp_already and "order book" in MODES:
        print("the request's materials already plan from the order book (MRP) — one answer covers both")
        MODES = {"as is": False}
    ob = T["outbound"]
    row = {"customer_id": REQ, "product_id": ITEM, "volume": 1e-6, "time_unit": "week",
           "unit_price": products[ITEM].get("unit_price"), "expected_lead_time": TRANSIT}
    data_req = with_tables(data, outbound=pd.concat([ob, pd.DataFrame([row])], ignore_index=True))

    def with_order(schedule, order_book=False):
        """The policy copy with the request row's firm delivery schedule (units per week from week 0,
        waiting if unmet) and, for `order_book`, the request's materials planned by MRP on every lane
        that supplies them."""
        snap = copy.deepcopy(pol.snapshot)
        over = snap.setdefault("overrides", [])
        patch = {"backorder_allowed": True, "max_backorder_days": 182}
        if SERVE_LAST:
            patch["row_priority"] = 0
            if rule not in ("priority", "sla_tier"):
                snap.setdefault("defaults", {}).setdefault("fulfillment", {})["allocation"] = "priority"
        over.extend([
            {"scope": "node", "target_key": f"{REQ}::{ITEM}", "family": "demand",
             "patch": {"row_demand_schedule": [float(x) for x in schedule]}},
            {"scope": "node", "target_key": f"{REQ}::{ITEM}", "family": "fulfillment", "patch": patch}])
        if order_book:
            for sup, mat in links[links.material_id.isin(MATS)][["supplier_id", "material_id"]].itertuples(index=False):
                key = f"{sup}::{mat}"
                hit = next((o for o in over if o.get("target_key") == key and o.get("family") == "inventory"), None)
                if hit:
                    hit["patch"] = {**(hit.get("patch") or {}), "type": "mrp"}
                else:
                    over.append({"scope": "node", "target_key": key, "family": "inventory", "patch": {"type": "mrp"}})
        return Policy(id=pol.id, policy_hash=None, snapshot=snap, label="with the request")

    if SERVE_LAST and rule not in ("priority", "sla_tier"):
        print(f"your allocation rule is {rule or 'the engine default'}; for this check the request is served after "
              "every existing row (priority rule, the request last) — both sides of the comparison use it")
    n_runs = len(SEEDS) * (len(WEEKS) + 2) * len(MODES)
    print(f"{n_runs} one-replication runs — about {ONE_RUN * n_runs:.0f}s")
    for mode, order_book in MODES.items():
        ZEROS[mode] = zero = {s: inspect_run(data_req, with_order([0.0], order_book), FRAME_ALL, PROJECT_MODEL, s)[1]
                              for s in SEEDS}

        def delivered(f, s, zero=zero):
            """Units delivered to the request by each week: the product's extra deliveries plus what its
            existing customers lost to it, capped at the quantity."""
            dF = (f[("product", "fulfillment")][ITEM] - zero[s][("product", "fulfillment")][ITEM]).cumsum()
            dL = (f[("product", "lost_units")][ITEM] - zero[s][("product", "lost_units")][ITEM]).cumsum()
            return (dF + dL).clip(upper=QUANTITY)

        rows, TRIALS[mode] = [], {}
        for w in WEEKS:
            worlds = {s: inspect_run(data_req, with_order([0.0] * w + [QUANTITY], order_book), FRAME_ALL,
                                     PROJECT_MODEL, s)[1] for s in SEEDS}
            TRIALS[mode][w] = worlds
            harm = pd.Series({s: max(0.0, sum((f[("product", "lost_units")][p] - zero[s][("product", "lost_units")][p]).sum()
                                              for p in f[("product", "lost_units")].columns)) for s, f in worlds.items()})
            rows.append(candidate_row(w, pd.DataFrame({s: delivered(f, s) for s, f in worlds.items()}), harm))
        CANDS[mode] = pd.DataFrame(rows).set_index("week")

        # A split delivery, when the plant cannot make it all in one week: equal parts in the weeks up to
        # the wanted one, as many as the capacity existing customers leave free requires.
        cap = products[ITEM].get("production_capacity") or np.inf
        spare = pd.Series({s: cap - f[("product", "production")][ITEM].loc[SHIP_WEEK] for s, f in zero.items()}).quantile(q)
        if CANDS[mode].loc[SHIP_WEEK, "on time"] < CONFIDENCE and 0 < spare < QUANTITY:
            k = min(int(np.ceil(QUANTITY / spare)), SHIP_WEEK + 1, SEARCH_WEEKS + 1)
            plan = [0.0] * (SHIP_WEEK - k + 1) + [QUANTITY / k] * k
            worlds = {s: inspect_run(data_req, with_order(plan, order_book), FRAME_ALL, PROJECT_MODEL, s)[1] for s in SEEDS}
            got = pd.DataFrame({s: delivered(f, s) for s, f in worlds.items()})
            SPLITS[mode] = {"parts": k, "each": QUANTITY / k, "from": SHIP_WEEK - k + 1,
                            "on time": got.loc[SHIP_WEEK].ge(QUANTITY - TOL).mean()}
else:
    oh, lt = stock(WORLDS, ITEM), lead_weeks(ITEM) or 2
    free = free_stock(oh, lt).clip(upper=QUANTITY)
    CANDS["from stock"] = pd.DataFrame([candidate_row(w, free, None) for w in range(H)]).set_index("week")

for mode, cand in CANDS.items():
    shown = cand.loc[[w for w in WEEKS if w in cand.index]]
    display(styled(shown.assign(**{"on time": shown["on time"] * 100}).rename(columns={"on time": "on time (% of futures)"}),
                   fmt={"on time (% of futures)": "{:.0f}", f"on time at {CONFIDENCE:.0%}": "{:,.0f}",
                        f"complete by (at {CONFIDENCE:.0%})": "{:.0f}", "others lose (median)": "{:,.0f}"},
                   shade=["on time (% of futures)"], cmap="Greens",
                   caption=f"{QUANTITY:,.0f} × {ITEM}, wanted in week {SHIP_WEEK} — materials {mode}"))

## 5 · The answer
**Expect:** yes, or how much on time and when the rest, and the nearest week each way that works.

In [ ]:
def week_text(w):
    return f"week {int(w)} ({to_date(int(w))})"


what = "ship" if REQUEST_KIND == "product" else "issue"


def answer(cand, mode):
    """The planner's answer for one way of replenishing: yes, or partial + when, pull in, push back."""
    at = cand.loc[SHIP_WEEK]
    ok_weeks = [w for w in cand.index if cand.loc[w, "on time"] >= CONFIDENCE]
    pull = max([w for w in ok_weeks if w < SHIP_WEEK], default=None)
    push = min([w for w in ok_weeks if w > SHIP_WEEK], default=None)
    on_time_qty, complete = at[f"on time at {CONFIDENCE:.0%}"], at[f"complete by (at {CONFIDENCE:.0%})"]
    harm = at["others lose (median)"]
    cost = (f" Existing customers lose a median **{harm:,.0f}** units to it." if harm >= TOL else
            " Existing customers lose nothing to it.") if harm == harm else ""
    out = [f"#### Materials {mode}"]
    if at["on time"] >= CONFIDENCE:
        out.append(f"**Yes — promise it.** Complete by {week_text(SHIP_WEEK)} in {at['on time']:.0%} of futures.{cost}")
        if pull is not None:
            out.append(f"- It could even {what} from {week_text(pull)} (pull in).")
    else:
        out.append(f"**Not all of it on time** — complete by then in {at['on time']:.0%} of futures (the bar is "
                   f"{CONFIDENCE:.0%}).{cost}")
        if on_time_qty >= TOL:
            out.append(f"- **Partial:** {on_time_qty:,.0f} by {week_text(SHIP_WEEK)}, the remaining "
                       f"{QUANTITY - on_time_qty:,.0f} by " + (f"{week_text(complete)}." if complete == complete else
                                                               "a date beyond the horizon at this confidence."))
        later = push if push is not None else (complete if complete == complete else None)
        if later is not None:
            out.append(f"- **Push back:** all of it by {week_text(later)}.")
        if pull is not None:
            out.append(f"- **Pull in:** all of it by {week_text(pull)}, before the capacity or stock it needs is taken.")
        split = SPLITS.get(mode)
        if split:
            out.append(f"- **Split:** {split['parts']} deliveries of {split['each']:,.0f}, weeks {split['from']}–{SHIP_WEEK} "
                       f"(the first on {to_date(split['from'])}) — complete on time in {split['on time']:.0%} of futures. "
                       "The plant cannot make it all in one week beside existing orders.")
        if later is None and pull is None and not split:
            out.append("- **No week within the horizon works at this confidence.**")
        if REQUEST_KIND == "material":
            sup = links[links.material_id == ITEM].sort_values("lead_time_weeks").iloc[0]["supplier_id"]
            spare = suppliers[sup]["capacity_per_week"] - lanes[lanes.supplier_id == sup]["per_week"].sum()
            out.append(f"- **Order it:** a new order from {sup} placed now arrives about {week_text(lead_weeks(ITEM))} "
                       f"({sup}'s spare capacity is about {max(spare, 0):,.0f} / week).")
    return out


head = f"### {QUANTITY:,.0f} × {ITEM}, wanted {week_text(SHIP_WEEK)}" + (
    f" — at the customer {to_date(NEED_WEEK)}" if REQUEST_KIND == "product" else "")
answers = {mode: answer(cand, mode) for mode, cand in CANDS.items()}
if len(answers) == 2 and len({tuple(a[1:]) for a in answers.values()}) == 1:
    body = next(iter(answers.values()))[1:] + ["", "*The same either way: planning the request's materials from the "
                                               "order book changes nothing here — material is not what limits it.*"]
else:
    body = [line for a in answers.values() for line in a]
display(Markdown("\n".join([head] + body)))

span = WEEKS if REQUEST_KIND == "product" else list(range(max(0, SHIP_WEEK - 8), min(H, SHIP_WEEK + 9)))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
width = 0.8 / len(CANDS)
for i, (mode, cand) in enumerate(CANDS.items()):
    x = np.array(span) + (i - (len(CANDS) - 1) / 2) * width
    color = ["#2a78d6", "#008300"][i % 2]
    axes[0].bar(x, cand.loc[span, "on time"] * 100, width=width, color=color, label=f"materials {mode}")
    axes[1].bar(x, cand.loc[span, f"on time at {CONFIDENCE:.0%}"], width=width, color=color, label=f"materials {mode}")
axes[0].axhline(CONFIDENCE * 100, color="#d9480f", linewidth=1, linestyle="--", label=f"bar: {CONFIDENCE:.0%}")
axes[0].set_title(f"Complete on time, by {what} week (% of futures)", fontsize=10)
axes[1].axhline(QUANTITY, color="#d9480f", linewidth=1, linestyle="--", label="requested")
axes[1].set_title(f"Units on time in that week ({CONFIDENCE:.0%} of futures)", fontsize=10)
for ax in axes:
    ax.axvline(SHIP_WEEK, color="#495057", linewidth=0.8, linestyle=":")
    ax.set_xlabel("week")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 6 · What stands in the way
For a product request: how many units of it each constraint could cover in the wanted week — the plant's
capacity left over by existing customers, and each material's stock at the start of the week (with the
request in the books, so anything ordered for it counts) divided by what one unit uses. The smallest is
what limits the on-time quantity.
**Expect:** one table per way of replenishing, the binding constraint marked.

In [ ]:
if REQUEST_KIND == "product":
    cap = products[ITEM].get("production_capacity") or np.nan
    rate = need.set_index("material_id")["rate"]
    start = max(SHIP_WEEK - 1, 0)
    for mode, zero in ZEROS.items():
        covers = {f"plant capacity ({cap:,.0f}/wk)": pd.Series(
            {s: cap - f[("product", "production")][ITEM].loc[SHIP_WEEK] for s, f in zero.items()})}
        for m in MATS:
            covers[f"{m} stock (lead time {lead_weeks(m)} wk)"] = pd.Series(
                {s: f[("material", "on_hand")][m].loc[start] / rate[m] for s, f in TRIALS[mode][SHIP_WEEK].items()})
        tab = pd.DataFrame({"covers (median)": {k: v.median() for k, v in covers.items()},
                            f"covers (low, {q:.0%})": {k: v.quantile(q) for k, v in covers.items()}})
        tab["short of the request by"] = (QUANTITY - tab[f"covers (low, {q:.0%})"]).clip(lower=0)
        tab["binds"] = np.where(tab[f"covers (low, {q:.0%})"] == tab[f"covers (low, {q:.0%})"].min(), "yes", "")
        display(styled(tab.rename_axis(f"week {SHIP_WEEK}"), fmt={c: "{:,.0f}" for c in tab.columns if c != "binds"},
                       shade=["short of the request by"],
                       caption=f"Materials {mode}: units of {ITEM} each constraint covers in week {SHIP_WEEK}"))
    print("a material's stock can be topped up within the week by deliveries already on the way, so it can cover a\n"
          "little more than shown; plant capacity cannot.")
else:
    print("a material request is limited by its own stock and lead time — see step 3")

**Reading it.** A request that only fails on plant capacity can often ship partly now and the rest a week
later; one that fails on a material waits for that material's lead time, which is why its push-back date
jumps. "Others lose" above zero means the request, even served last, used capacity or material an existing
customer needed later: promising it is a choice, not a free answer. When the "order book" answer is better
than "as is", the gap is purchasing not seeing the order: telling them now (or running those materials on
MRP) is what closes it. Raise `N_WORLDS` before you quote a date that sits just above the bar.

## 7 · Export, and what produced this answer
**Expect:** the record, and the workbook written (in Colab, downloaded).

In [ ]:
record = pd.Series({
    "project": f"{project['name']} ({PID})", "supply chain model": PROJECT_MODEL,
    "dataset version": f"{data.id} · v{data.version_no} · graph_hash {data.graph_hash}",
    "policy version": f"{pol.label} ({pol.id}) · policy_hash {pol.policy_hash}",
    "request": f"{REQUEST_KIND}: {QUANTITY:,.0f} × {ITEM}, week {SHIP_WEEK}"
               + (f" (+{TRANSIT} wk transit)" if REQUEST_KIND == "product" else ""),
    "week 0": str(START.date()), "futures": f"{len(SEEDS)} (seeds {SEEDS[0]}–{SEEDS[-1]}) · confidence {CONFIDENCE:.0%}",
    "served": ("after every existing row" if SERVE_LAST else "by the project's rule") if REQUEST_KIND == "product" else "from stock",
    "materials": ", ".join(CANDS),
    "computed by": "this machine (client)",
}, name="value")
display(record.to_frame())
if EXPORT_XLSX:
    with pd.ExcelWriter(EXPORT_XLSX) as xw:
        record.to_frame().to_excel(xw, sheet_name="record")
        POSITION.to_excel(xw, sheet_name="stock position")
        for mode, cand in CANDS.items():
            cand.to_excel(xw, sheet_name=f"weeks · materials {mode}"[:31])
    print("written:", EXPORT_XLSX)
    try:
        from google.colab import files
        files.download(EXPORT_XLSX)
    except ImportError:
        pass

## Next
- **04** — every material week by week, and what each supplier outage does to them.
- **03** — try the fix on a local copy first: a second supplier lane, more safety stock, more capacity.